# EBD-DNS — widening the empirical base for the architectural floor

**Companion notebook to** *Exfiltrated Bytes before Detection: Damage-Denominated Evaluation
and the Architectural Floor of Record-Consuming Exfiltration Detectors.*

The manuscript measures the floor on **n = 7** episodes from one semi-synthetic dataset;
Limitation 2 concedes the "five of seven at floor" headline carries a 95% interval of
roughly [0.36, 0.95]. This notebook measures the same quantities on an independent,
**packet-level** corpus: nine exfiltration tools in eight families, captured in three
different labs.

## Run it in two passes

**First, attach the data.** Add Input → Datasets → search *daumel dns-tunneling-dataset*
("DNS exfiltration dataset") → Add. Importing this `.ipynb` into Kaggle does not bring its
inputs with it; without the dataset the discovery cell stops and says so.

1. **`SMOKE = True`** (the default) — about 3–5 minutes. Validates the engine, prints the corpus
   layout it discovered, and runs everything on one capture per tool and one university
   capture. It is plumbing, not evidence: with one university capture its false-alarm
   numbers are in-sample.
2. **`SMOKE = False`**, then **Save Version → Save & Run All** (so it keeps running with the
   tab closed). The measurement run: all 18 usable tool captures, all 68 top-1M captures and
   all 56 university captures (the first 300k packets of each), memory permitting. Plan on
   roughly 15–20 minutes.
   Memory: a synthetic replica at the real packet counts (6.1 million benign queries) took 12.5
   minutes and peaked at 4.2 GB, but a replica is not the real thing and real captures can cost
   more per query. So the notebook reads the container's own memory limit, **prints where that
   number came from**, measures what the first university capture really costs, and if the whole
   corpus would not leave room for the units, features and training that follow, it reads an
   evenly spaced subset of the university captures instead and says so (MEMORY GUARD, printed by
   the parse cell and repeated in the digest). If the container does not show its limit, the
   notebook plans for 16 GB, the smallest figure ever quoted for a Kaggle CPU session (it does the same
   with a figure above 40 GB, which is probably the host's); the first line of output says so and
   calls it unconfirmed, and `MEMORY_LIMIT_GB` in the CONFIGURATION cell overrides it once you
   know the session's real figure. Every phase prints a `[memory]` line, and the floor tables are
   saved as soon as they are computed. The cell timings and peak memory kept in
   `dns_provenance.json` say where the time and the memory actually went.

**What changed in v4.2.** The 7 Oct measurement run (v4.1) showed four defects on the benign
side. None touches a floor: floors use no benign data and come out identical. Two independent
reviews of the build then found the further points noted under each item.

1. *Parser.* A name whose compression pointers loop made the decoder recurse until Python gave
   up, and the per-capture handler then ended the **whole capture**: four of the 13 university
   captures read on 7 Oct stopped that way. The decoder is now iterative, and a packet that
   raises is counted and skipped, never fatal. A capture ends early only if the container reader
   itself fails, and the parse cell then names the file and the packet it reached. The decoder
   also enforces two rules of RFC 1035 that v4.1 did not (a pointer cannot lead into the 12-byte
   header; a name is at most 255 octets): they turn only garbage away.
2. *Operating point.* v4.1 searched a threshold grid whose lowest value, 0.005, was already inside
   the budget in all 54 folds, so the 1% budget never bound and every detector ran at its most
   sensitive setting. v4.2 sets the threshold exactly, from the benign scores themselves. The
   digest says what set the threshold in each fold: the budget itself, a tie plateau (equal scores
   at the edge of the budget, so it cannot be spent in full), the model's own zero scores, or too
   few benign units to certify the budget at all (under 381 at 1%: nothing alerts). It says "the
   budget was binding" only for the first, and gives the share of the budget the training units used.
3. *Memory.* The digest says where the limit came from and prints the raw readings. The 16.0 GB of
   the 7 Oct run equals the fallback constant exactly, so the limit was either unreadable or
   discarded: v4.1 also discarded any figure above 32 GB, and a genuine 30 GiB limit is 32.2 GB.
   v4.2 accepts figures up to 40 GB and labels a fallback "assumed, unconfirmed".
4. *DNS over TCP.* A message whose 2-byte length arrived in a segment of its own was read from
   the wrong offset; it is now recovered, but only when it starts at the sequence number right after
   the length, is exactly as long as announced and has one question. A stray 2-byte segment can no
   longer turn the next ordinary segment into an empty name.

The digest also gives **family-level** numbers: a family-weighted mean with two intervals, a
two-stage percentile bootstrap (at 8 families it covers about 90%, not 95%, in simulation) and a
Student-t interval over the family means (about 95%), and the families with total loss as k/8 with
Wilson and exact intervals. It labels a benign source that has a single capture as in-sample, and
reports the parser's error counts.

**What to send back:** `ebd_dns_results.zip` from the Output tab. If downloading is awkward,
copy the **RESULTS DIGEST** block printed by the last code cell — it carries every
headline number.

**If a cell stops,** its message names the cause and the table to send back. Discovery,
parsing and episode building each print a per-file account *before* they stop, so the
output above the error is the diagnosis. If the kernel itself is killed with no message, it ran
out of memory (the first v4 measurement run did, in cell 12; v4.1 added the memory guard). The
`[memory]` lines above the crash show how far it got and how full the container was, and the floor
tables (`dns_floors.csv`, `dns_cadence_sweep.csv`, `dns_parse_stats.csv`,
`dns_malformed_samples.csv`) are already in the Output folder: send those back. To run again with
less, set `N_UNIVERSITY_FILES = 28` in the CONFIGURATION cell (it halves the university corpus), or
set `MEMORY_LIMIT_GB` to the limit this session really has so that the guard plans for it, and say
so when you send the results back. If the session has MORE memory than the notebook assumed (the
first output line says what it assumed and why), setting `MEMORY_LIMIT_GB` lets the guard read more
of the university captures.

## What this measures that DAPT2021 cannot

The manuscript's family claim is that the binding constraint on early exfiltration
detection is **pipeline position, not model quality**. Here three consumers run over
**identical packets** — per-query, flow-record, session — with channel, adversary, asset,
model family, features and operating-point rule held constant. Only the unit of inference
varies.

A fourth detector, an **ibHH-style streaming accumulator** (after Ozery et al.,
*Information-Based Heavy Hitters*), sits outside the record-consuming family. Its excess
over the per-query floor is **statistical** delay — evidence accumulation — rather than
architectural delay. That separates the two components the manuscript's quickest-change-
detection discussion says compose, and it is calibrated on benign traffic only, so its
operating point never sees the traffic it is scored on.

## Corpus — `daumel/dns-tunneling-dataset`

| | Captures used | Source | Why |
|---|---|---|---|
| **Malicious, `_up`** | 15 files, 7 tools | Chen et al. (cobaltstrike, dns2tcp, dnscat2, iodine, ozymandns); dataset author (dnsexfiltrator ×2) | upstream transfer = exfiltration; payload rides in QNAMEs |
| **Malicious, no direction** | `dnspot`, `tcp-over-dns` ×2 | Gao et al. (GraphTunnel) | included, flagged, reported separately |
| **Malicious, `_down`** | 13 files — **excluded** | Chen et al. | data flows *into* the network; not exfiltration, wrong denominator |
| **Benign, top-1M** | 68 files | Gao et al. — lookups of popular domains (names with a subdomain only) | |
| **Benign, university** | 56 hourly 2016 captures, ~30 GB; the first 300k packets (~10 min) of each are read | Singh et al. (TI-2016) | real campus DNS — the realistic false-positive baseline, one distinct day/hour per file |

Three labs means three capture set-ups (link types, vantage points); the parser reads every
common link type and reports any it cannot read rather than guessing.

**Capture length is part of the data.** Four of the five Chen et al. tool captures hold exactly
50,000 packets (25,000 queries) and the fifth 100,000, so for those tools the "transfer" is
whatever the first 25,000 queries carried and its duration $D$ is how long that took (53–177 s).
A flow-record total loss on such a capture means *a transfer of that size and speed* is lost
before any record exists; it is not a statement about how long the real exfiltration would
have run. The active-timeout sweep below shows how that conclusion moves with the exporter's
timer.

`dnsexfiltrator` and `dnsexfiltrator_modified` are one family for the leave-one-tool-out
split, so a variant can never leak its parent's signature.

**Expect tens of episodes, not hundreds.** Each capture is one tool run; long runs with
pauses split into several episodes. Roughly 18–60 episodes across eight families is the
realistic range. The widening this buys is as much **tool diversity** as *n*.

## Method

**The floor.** For a consumer whose unit of inference is a completed record, the earliest
possible alert is the emission of its first record. With an oracle detector (A4, correct on
its first available record) the fraction of the asset already gone at that moment is

$$\mathrm{EBD}_{\text{floor}} \;=\; \frac{V_e\!\left(t_0 + T_{\text{eff}}\right)}{V_e(\text{total})}
\;\;\ge\;\; \min\!\left(1,\; T_{\text{eff}}/D\right)\ \text{under constant rate}$$

The closed form is reported beside the measured value, never instead of it.

**Episodes.** Scoped to their capture file — lab captures reuse client IPs across files, and
grouping across files would merge independent tool runs. Within a file, queries are grouped
by (client, registered domain) and only the **dominant** domain is kept: every capture is
one tool run against one tunnel domain, and the rest of the file is background DNS. If a
capture saw both the stub→resolver and resolver→authoritative legs, the transfer is counted
once. Every one of these decisions is logged and exported.

**Payload unit (A5).** EBD is a ratio, so encoded QNAME characters stand in exactly for
decoded bytes when each query's characters are proportional to its bytes. Per-query
protocol overhead breaks that, so every floor is re-measured with up to half of a typical
query assumed to be overhead, and the difference is reported. The unit is measured, not
assumed.

**Features are causal.** Detector features use only what a deployed detector could compute
from the record in hand: each name is split against its registered domain, never against a
suffix inferred from the whole episode. EBD accounting uses the full-episode payload —
EBD is an offline metric, as the manuscript states.

**Operating point.** The manuscript's revised a-priori rule: the most aggressive threshold
whose **upper** 95% Wilson bound on the training-benign burden sits inside the budget. Fixed
before any held-out data is touched. Since v4.2 the rule is exact: the budget allows at most
$k$ of the $n$ training-benign units to alert, and the threshold is the lowest observed score
that leaves the $(k+1)$-th highest one silent. The digest reports, per consumer, what set the
threshold in each fold (the budget, a tie plateau, the model's own zeros, or too few benign units)
and how much of the budget the training units used.

**Split.** Leave-one-family-out for malicious; benign split by capture file, stratified by
source. Burden is reported per benign source, and as **alerts per monitored hour** — the
only form comparable across granularities.

**Assumptions carried from the manuscript.** (A2) a record covering traffic up to $L$ cannot
be emitted before $L$ — enforced in `export_records`. (A4) oracle floor. Session records are
credited with the earliest emission physically possible, so every session figure is a
**lower bound**.

In [ ]:
# ============================== CONFIGURATION ==============================
VERSION = "4.2"
SMOKE = True        # 1st pass: engine self-test + layout check + one capture per tool
FULL  = False       # read each university capture to its END (~30 GB) instead of its first
                    # UNI_PACKET_CAP packets. Not needed: units are capped anyway (MAX_BENIGN_UNITS).

DATA_ROOT = "/kaggle/input"
OUT_DIR   = "/kaggle/working"

DIRECTIONS_INCLUDED = ("up", "unspecified")  # '_down' = data flowing IN: not exfiltration
N_UNIVERSITY_FILES  = 10**9       # ALL 56 university captures (a distinct day/hour each); SMOKE uses 1
UNI_PACKET_CAP      = 300_000     # first 300k packets of each (~10 min of campus DNS)

# Episodes
GAP_GRID, GAP_PRIMARY = [60.0, 300.0, 600.0], 300.0
MIN_QUERIES, MIN_PAYLOAD_CHARS = 20, 512

# Telemetry cadence. Vendor defaults span 30x: Juniper flow-active-timeout 60 s,
# Cisco IOS Flexible NetFlow and IOS XR 1800 s, CICFlowMeter 120 s.
FLOW_ACTIVE_TIMEOUT = 1800.0
IDLE_GRID, IDLE_PRIMARY = [5.0, 15.0, 30.0, 60.0, 120.0], 15.0
SESSION_GRID, SESSION_PRIMARY = [60.0, 300.0, 600.0, 1800.0], 300.0
ACTIVE_GRID = [30.0, 60.0, 120.0, 300.0, 600.0, 1800.0]   # the vendor-default span, swept at idle = IDLE_PRIMARY

# Detectors
FPB_BUDGET = 0.01                   # 1% of benign units, upper Wilson bound
MAX_BENIGN_UNITS = 400_000          # per granularity and split; deterministic sample, logged
MAX_TRAIN_UNITS_PER_EPISODE = 5_000 # training only; held-out episodes are scored in full
RUN_IBHH, IBHH_WINDOW_S = True, 60.0
SEED, N_BOOT = 20260825, 10_000

# Memory. Kaggle's session RAM is quoted as 16 GB, 29 GB or 30 GB depending on the source and the date, and was not
# verified; the figure on the session panel is the one that counts. The notebook reads the container's own limit
# and, if the benign corpus would not leave room for the units, features and training that follow the
# parse, reads FEWER university captures (an evenly spaced subset) and says so in the digest. It never
# dies silently: the first v4 measurement run on Kaggle did, in the training cell.
MEMORY_LIMIT_GB = None              # None = detect (cgroup); a number overrides the detected limit
MEM_FRAC_PARSE  = 0.30              # share of the limit the parsed corpus may use. Everything after the parse
                                    # (units, features, training) added about 0.7x that again on a synthetic replica at
                                    # the real packet counts, so a corpus at its budget peaks near half the limit
MIN_UNI_FILES   = 8                 # never plan fewer university captures than this

AT_FLOOR_TOL_PP = 0.01              # "within noise of the floor": the A5 payload-unit uncertainty is of
                                    # this order, so a gap below it is not a measurable shortfall.
                                    # The strict rule (|EBD - floor| < 1e-9) stays the primary count.
MAX_PACKETS_BENIGN = UNI_PACKET_CAP # malicious captures are NEVER truncated (it would bias floors)
if SMOKE:
    N_UNIVERSITY_FILES = 1
if FULL:
    MAX_PACKETS_BENIGN = None

import os, sys, re, math, json, time, random, zipfile, warnings
from collections import Counter, defaultdict
os.makedirs(OUT_DIR, exist_ok=True)
warnings.filterwarnings("ignore")
random.seed(SEED)
RUN_STARTED = time.strftime("%Y-%m-%d %H:%M:%S")

# Per-cell wall clock (Jupyter only). v3 printed no breakdown, so a 31-minute smoke run could not be
# explained from its outputs. This lands in dns_provenance.json and in the digest.
CELL_SECONDS, CELL_PEAK_MB = [], []

def _peak_mb():
    """High-water resident memory of this kernel so far, in MB (None if the OS does not say)."""
    try:
        import resource
        v = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss          # KB on Linux, bytes on macOS
        return round(v / (1024 * 1024 if sys.platform == "darwin" else 1024))
    except Exception:
        return None

try:
    _ip = get_ipython()
    def _t_pre(info=None):
        global _CELL_T
        _first = ((getattr(info, "raw_cell", "") or "").strip().splitlines() or [""])[0]
        _CELL_T = (time.time(), re.sub(r"[=\s]+$", "", re.sub(r"^[#=\s]+", "", _first))[:70])   # "# ==== NAME ====" -> "NAME"
    def _t_post(result=None):
        try:
            CELL_SECONDS.append((_CELL_T[1], round(time.time() - _CELL_T[0], 1))); CELL_PEAK_MB.append(_peak_mb())
        except NameError: pass            # the CONFIGURATION cell itself: its pre-event was missed
    if not getattr(_ip, "_ebd_timer", False):          # register once, even if this cell is re-run
        _ip._ebd_timer = True
        _ip.events.register("pre_run_cell", _t_pre); _ip.events.register("post_run_cell", _t_post)
except Exception:
    pass                                  # not in Jupyter: the timings are simply absent

def _read_int(path):
    try:
        v = open(path).read().strip()
        return None if v in ("", "max") else int(v)
    except Exception:
        return None

KAGGLE_FALLBACK_BYTES = 16e9     # the smallest RAM figure ever quoted for a Kaggle CPU session
KAGGLE_PLAUSIBLE_MAX_BYTES = 40e9   # no Kaggle CPU session is thought to have more than 30 GiB (32.2 GB); a larger figure is the host's

def _mem_limit_info():
    """-> (limit in bytes or None, where the number came from, the raw readings).

    v4.1 returned the number alone. The 7 Oct digest said 16.0 GB, which is exactly the Kaggle fallback below (a
    16 GiB cgroup would have printed 17.2), so the run could not say whether it had READ its limit or assumed it,
    and the memory guard planned the university slice from that figure: 13 of 56 captures. The source and the raw
    readings are printed at the top of the run and kept in the digest and in dns_provenance.json."""
    raw = {"cgroup_v2_memory.max": None, "cgroup_v1_limit_in_bytes": None, "MemTotal_bytes": None,
           "kaggle_dir": os.path.isdir("/kaggle")}
    for key, path in (("cgroup_v2_memory.max", "/sys/fs/cgroup/memory.max"),
                      ("cgroup_v1_limit_in_bytes", "/sys/fs/cgroup/memory/memory.limit_in_bytes")):
        try:
            raw[key] = open(path).read().strip()
        except Exception:
            pass                                             # not there: stays None
    try:
        for line in open("/proc/meminfo"):
            if line.startswith("MemTotal:"):
                raw["MemTotal_bytes"] = int(line.split()[1]) * 1024
    except Exception:
        pass
    if MEMORY_LIMIT_GB:
        return int(MEMORY_LIMIT_GB * 1e9), "MEMORY_LIMIT_GB set in the CONFIGURATION cell", raw
    c = []
    for key, name in (("cgroup_v2_memory.max", "cgroup v2 memory.max"),
                      ("cgroup_v1_limit_in_bytes", "cgroup v1 memory.limit_in_bytes")):
        try:
            v = int(raw[key])
        except (TypeError, ValueError):
            continue                                         # unreadable, or the word "max"
        if 0 < v < 2 ** 50:                                  # "unlimited" shows up as a huge number
            c.append((v, name))
    if c:
        lim, how = min(c)
    elif raw["MemTotal_bytes"]:
        lim, how = raw["MemTotal_bytes"], "MemTotal (no cgroup limit is set)"
    else:
        lim, how = None, "unknown"
    if raw["kaggle_dir"] and (lim is None or lim > KAGGLE_PLAUSIBLE_MAX_BYTES):
        # v4.1 cut at 32e9, which also threw away a genuine 30 GiB limit (32.2e9), and then claimed the figure was the
        # host's although it had been read from the container's own cgroup. Nothing here can tell which it was.
        seen = (f"{lim / 1e9:.1f} GB read from {how} is more than any Kaggle session is thought to have, so it may be the "
                f"host's" if lim else "no limit could be read")
        return (KAGGLE_FALLBACK_BYTES,
                f"FALLBACK: {KAGGLE_FALLBACK_BYTES / 1e9:.0f} GB assumed for a Kaggle session, unconfirmed ({seen})", raw)
    return lim, how, raw

def _mem_limit_bytes():
    return _mem_limit_info()[0]

def _mem_used_bytes():
    """Memory that cannot simply be dropped: anonymous + shared (+ kernel), container-wide where the cgroup
    says so (file cache is excluded: the kernel gives it back), else this process's resident set."""
    for stat, keys in (("/sys/fs/cgroup/memory.stat", ("anon", "shmem", "kernel_stack", "slab_unreclaimable")),
                       ("/sys/fs/cgroup/memory/memory.stat", ("total_rss", "total_shmem"))):
        try:
            d = dict(line.split()[:2] for line in open(stat))
            return sum(int(d.get(k, 0)) for k in keys)
        except Exception:
            pass
    try:
        return int(open("/proc/self/statm").read().split()[1]) * os.sysconf("SC_PAGE_SIZE")
    except Exception:
        return None

MEM_LOG = []                                  # (phase, GB in use) -> dns_provenance.json
def memory_mark(phase):
    """Record, and print, how much memory is in use after a phase. If the kernel is ever killed, the last of
    these lines above the crash say how far the run got and how full the container already was."""
    u = _mem_used_bytes()
    if u:
        MEM_LOG.append((phase, round(u / 1e9, 2)))
        lim = _mem_limit_bytes()
        print(f"  [memory] {phase}: {u / 1e9:.2f} GB in use" + (f" of {lim / 1e9:.1f} GB" if lim else ""), flush=True)
    return u

def _release():
    """Hand freed memory back: collect garbage, then ask glibc to return what it holds."""
    import gc
    gc.collect()
    try:
        import ctypes
        ctypes.CDLL("libc.so.6").malloc_trim(0)
    except Exception:
        pass

import pandas as pd, numpy as np
ORDER = ["per-query", "flow", "session"]      # one fixed order: tables, folds, figures
KW = dict(t_active=FLOW_ACTIVE_TIMEOUT, t_idle=IDLE_PRIMARY, window_s=SESSION_PRIMARY)

def wfmt(k, n):
    lo, hi = wilson(k, n); return f"[{lo*100:.1f}, {hi*100:.1f}]"

def efmt(k, n):                                  # the exact (Clopper-Pearson) interval, for counts as small as 3 of 8 families
    lo, hi = clopper_pearson(k, n); return f"[{lo*100:.1f}, {hi*100:.1f}]"

def _pc(x, nd=3):                                # a percentage, or "n/a" where there was nothing to measure (no held-out units)
    return "n/a" if x != x else f"{x * 100:.{nd}f}%"

def _ph(x, unit="/h"):
    return "n/a" if x != x else f"{x:.1f}{unit}"

def _fam(fn, by_f, label, **kw):                 # a family-level interval; a bad value is reported, not allowed to end the run
    try:
        return fn(by_f, **kw)
    except ValueError as e:
        print(f"  family-level interval skipped ({label}): {e}")
        return (float("nan"),) * 3

# Which cell creates what -- so running a cell before its inputs exist names the cell
# to run, instead of a bare NameError (e.g. after a kernel restart).
_MADE_BY = {**{k: "CORPUS DISCOVERY" for k in ("USE_MAL", "USE_BEN", "uni", "EXCLUDED_DOWN")},
            **{k: "PARSE CAPTURES" for k in ("PARSE", "mal_queries", "ben_queries", "bad_files", "trunc", "MEM_GUARD")},
            **{k: "EPISODES" for k in ("EPISODES", "EP_LOG", "episodes_by_gap")},
            **{k: "FLOORS" for k in ("floors", "summary", "notes", "A5_DELTA", "PQ_MEAN", "FLOW_MEAN", "FAMILY_FLOORS")},
            **{k: "measured floor vs the closed form" for k in ("CLOSED_EXC", "CLOSED_BELOW")},
            **{k: "CADENCE SWEEP" for k in ("sweep", "save_floor_tables")},
            **{k: "CAUSAL FEATURES" for k in ("BEN_TRAIN", "BEN_TEST", "HOURS", "IN_SAMPLE", "matrix",
                                              "unit_records", "featurise")},
            **{k: "LEAVE-ONE-FAMILY-OUT" for k in ("attain", "opp", "MODEL")},
            **{k: "ibHH-STYLE" for k in ("ibhh", "IBHH_META")},
            **{k: "ATTAINMENT RESULTS" for k in ("att_summary", "srcs", "over", "FAMILY_ATT", "OVER_SRC")},
            "DIGEST": "EXPORT + DIGEST"}

def _require(*names):
    missing = [n for n in names if n not in globals()]
    if missing:
        need = sorted({_MADE_BY.get(n, "?") for n in missing})
        raise RuntimeError(
            "This cell needs cells that have not run in this kernel: "
            + ", ".join(f"'{c}'" for c in need) + f"  (missing: {', '.join(missing)}). "
            "Use Run All, or right-click this cell > Run before. "
            "After a kernel restart, everything from the top has to run again.")

MEM_LIMIT_INFO = _mem_limit_info()
print(f"EBD-DNS notebook v{VERSION}   SMOKE={SMOKE}  FULL={FULL}  seed={SEED}  started {RUN_STARTED}")
print("memory limit: " + (f"{MEM_LIMIT_INFO[0] / 1e9:.1f} GB" if MEM_LIMIT_INFO[0] else "unknown") + f"   [{MEM_LIMIT_INFO[1]}]")
print("  raw readings: " + ", ".join(f"{k}={v}" for k, v in MEM_LIMIT_INFO[2].items()))
if MEM_LIMIT_INFO[1].startswith("FALLBACK"):
    print("  The container does not show its own limit, so the memory guard plans for the fallback. If this session has more\n"
          "  (the Kaggle session panel shows its RAM), set MEMORY_LIMIT_GB above and the guard will read more captures.")

### Engine

Pure standard library — Kaggle sessions often run offline, and a dependency that cannot install is a way for a long run to die at the start.

In [ ]:
from __future__ import annotations   # ===== ebd_dns_engine.py =====
"""
EBD-DNS engine — architectural floor and detector attainment for DNS exfiltration.

Companion to "Exfiltrated Bytes before Detection: Damage-Denominated Evaluation
and the Architectural Floor of Record-Consuming Exfiltration Detectors".

The manuscript measures the floor on n = 7 episodes from one dataset. This engine
measures the same quantity on an independent, packet-level corpus of DNS
exfiltration, and runs consumers at different pipeline positions -- per-query,
flow-record, session -- over IDENTICAL traffic, so only the unit of inference
varies. A streaming accumulator in the style of ibHH (Ozery et al.) adds a fourth,
outside-the-family point: its excess over the per-query floor is STATISTICAL delay
(evidence accumulation), not architectural delay -- the two components the
manuscript's QCD discussion says compose.

Payload unit (A5). EBD is a ratio V(t)/V(total). Encoded QNAME characters are an
exact stand-in for decoded bytes when each query's characters are proportional to
its bytes. Per-query protocol overhead (sequence numbers, session ids) breaks
exact proportionality, so the engine re-measures every floor with an estimated
per-query overhead removed and reports the difference. The unit is measured, not
assumed.
"""


import bisect
import math
import random
import struct
import sys
from collections import Counter, defaultdict, deque
from dataclasses import dataclass, field
from typing import Iterable, Sequence

# --------------------------------------------------------------------------- #
# Statistics                                                                    #
# --------------------------------------------------------------------------- #

def wilson(k: int, n: int, z: float = 1.959963985) -> tuple[float, float]:
    """Two-sided Wilson score interval for a binomial proportion, as fractions."""
    if n == 0:
        return (0.0, 1.0)
    p = k / n
    d = 1.0 + z * z / n
    centre = (p + z * z / (2 * n)) / d
    half = (z / d) * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n))
    return (max(0.0, centre - half), min(1.0, centre + half))


def wilson_upper(k: int, n: int, z: float = 1.959963985) -> float:
    return wilson(k, n, z)[1]


def bootstrap_ci(xs: Sequence[float], n_boot: int = 10_000, seed: int = 20260825,
                 alpha: float = 0.05) -> tuple[float, float]:
    """Percentile bootstrap CI for the mean. Returns (lo, hi)."""
    xs = list(xs)
    if len(xs) < 2:
        return (float("nan"), float("nan"))
    rng = random.Random(seed)
    n = len(xs)
    means = sorted(sum(xs[rng.randrange(n)] for _ in range(n)) / n for _ in range(n_boot))
    return (means[int((alpha / 2) * n_boot)], means[min(n_boot - 1, int((1 - alpha / 2) * n_boot))])


def _family_lists(by_family: dict) -> list[list[float]]:
    """The non-empty families as lists of floats, in a fixed order (sorted by key), so that a result never depends on
    the order the dict was built in. A NaN or infinite value is an upstream bug and stops here, naming the family."""
    out = []
    for key in sorted(by_family, key=str):
        v = [float(x) for x in by_family[key]]
        if not v:
            continue
        if not all(math.isfinite(x) for x in v):
            raise ValueError(f"non-finite value in family {key!r}")
        out.append(v)
    return out


def family_bootstrap(by_family: dict, n_boot: int = 10_000, seed: int = 20260825,
                     alpha: float = 0.05) -> tuple[float, float, float]:
    """Two-stage cluster bootstrap of a mean over episodes that come in families.

    Episodes of one tool differ mainly by DNS record type, so n episodes is closer to the number of families than
    to n independent observations. Each draw resamples the families with replacement and then the episodes inside
    each sampled family with replacement. Returns (mean of the family means, lo, hi): the point estimate weights
    every family equally, which is the number to quote when one tool contributes four episodes and another one.

    Coverage (checked by simulation in the v4.2 review): with only 8 families the percentile interval is
    anti-conservative. Its nominal 95% covered the family-level mean 0.86 to 0.94 of the time (about 0.89 for the
    between-family spread of the 7 Oct data), so read it as roughly a 90% interval. `family_t_interval` is the
    alternative with the right coverage at this size. The result does not depend on the order of `by_family`."""
    fams = _family_lists(by_family)
    if len(fams) < 2:
        m = sum(fams[0]) / len(fams[0]) if fams else float("nan")
        return (m, float("nan"), float("nan"))
    point = sum(sum(v) / len(v) for v in fams) / len(fams)
    rng = random.Random(seed)
    k = len(fams)
    draws = []
    for _ in range(n_boot):
        tot = 0.0
        for _f in range(k):
            v = fams[rng.randrange(k)]
            tot += sum(v[rng.randrange(len(v))] for _e in range(len(v))) / len(v)
        draws.append(tot / k)
    draws.sort()
    return (point, draws[int((alpha / 2) * n_boot)], draws[min(n_boot - 1, int((1 - alpha / 2) * n_boot))])


# Student's t and the exact binomial interval, in the standard library (the engine has no dependencies). Both rest on
# the regularized incomplete beta function; the tests compare them with scipy to 1e-9 where scipy is installed.

def _betacf(a: float, b: float, x: float) -> float:
    """Continued fraction of the incomplete beta function (modified Lentz; Numerical Recipes 6.4)."""
    tiny = 1e-300
    qab, qap, qam = a + b, a + 1.0, a - 1.0
    c, d = 1.0, 1.0 - qab * x / qap
    d = 1.0 / (d if abs(d) > tiny else tiny)
    h = d
    for m in range(1, 1000):
        m2 = 2 * m
        for aa in (m * (b - m) * x / ((qam + m2) * (a + m2)),
                   -(a + m) * (qab + m) * x / ((a + m2) * (qap + m2))):
            d = 1.0 + aa * d
            d = 1.0 / (d if abs(d) > tiny else tiny)
            c = 1.0 + aa / c
            c = c if abs(c) > tiny else tiny
            de = d * c
            h *= de
        if abs(de - 1.0) < 1e-15:
            break
    return h


def _betai(a: float, b: float, x: float) -> float:
    """Regularized incomplete beta function I_x(a, b)."""
    if x <= 0.0:
        return 0.0
    if x >= 1.0:
        return 1.0
    front = math.exp(math.lgamma(a + b) - math.lgamma(a) - math.lgamma(b) + a * math.log(x) + b * math.log1p(-x))
    if x < (a + 1.0) / (a + b + 2.0):
        return front * _betacf(a, b, x) / a
    return 1.0 - front * _betacf(b, a, 1.0 - x) / b


def t_quantile(p: float, df: float) -> float:
    """The p-quantile of Student's t distribution with df degrees of freedom."""
    if not 0.0 < p < 1.0 or df <= 0:
        raise ValueError("t_quantile needs 0 < p < 1 and df > 0")
    if p == 0.5:
        return 0.0
    if p < 0.5:
        return -t_quantile(1.0 - p, df)
    tail = 1.0 - p                                       # P(T > t) for t > 0 is I_{df/(df+t^2)}(df/2, 1/2) / 2
    lo, hi = 0.0, 1.0
    while 0.5 * _betai(df / 2.0, 0.5, df / (df + hi * hi)) > tail and hi < 1e12:
        lo, hi = hi, hi * 2.0
    for _ in range(200):
        mid = 0.5 * (lo + hi)
        if 0.5 * _betai(df / 2.0, 0.5, df / (df + mid * mid)) > tail:
            lo = mid
        else:
            hi = mid
    return 0.5 * (lo + hi)


def _beta_quantile(p: float, a: float, b: float) -> float:
    lo, hi = 0.0, 1.0
    for _ in range(200):
        mid = 0.5 * (lo + hi)
        if _betai(a, b, mid) < p:
            lo = mid
        else:
            hi = mid
    return 0.5 * (lo + hi)


def clopper_pearson(k: int, n: int, alpha: float = 0.05) -> tuple[float, float]:
    """Exact (Clopper-Pearson) two-sided interval for k successes in n trials, as fractions. Wilson is the interval
    used elsewhere in this notebook; for counts as small as 3 of 8 families the exact interval is the conservative
    companion to quote beside it."""
    if n <= 0:
        return (0.0, 1.0)
    lo = 0.0 if k <= 0 else _beta_quantile(alpha / 2.0, k, n - k + 1)
    hi = 1.0 if k >= n else _beta_quantile(1.0 - alpha / 2.0, k + 1, n - k)
    return (lo, hi)


def family_t_interval(by_family: dict, alpha: float = 0.05,
                      clip: tuple | None = None) -> tuple[float, float, float]:
    """Mean of the family means with a Student-t interval over the families (df = families - 1).

    The unit of replication is the family, so this treats the family means as the sample. In the v4.2 review's
    coverage simulation (8 families, nominal 95%) it covered 0.95 to 0.97 where the percentile cluster bootstrap
    covered 0.86 to 0.94, at the price of an interval about a quarter wider. `clip=(lo, hi)` bounds it for
    quantities that cannot leave a range (a floor is a fraction); a bound may be None."""
    means = [sum(v) / len(v) for v in _family_lists(by_family)]
    k = len(means)
    if k == 0:
        return (float("nan"), float("nan"), float("nan"))
    m = sum(means) / k
    if k < 2:
        return (m, float("nan"), float("nan"))
    sd = math.sqrt(sum((x - m) ** 2 for x in means) / (k - 1))
    h = t_quantile(1.0 - alpha / 2.0, k - 1) * sd / math.sqrt(k)
    lo, hi = m - h, m + h
    if clip:
        if clip[0] is not None:
            lo = max(lo, clip[0])
        if clip[1] is not None:
            hi = min(hi, clip[1])
    return (m, lo, hi)


def shannon_entropy(s: str) -> float:
    if not s:
        return 0.0
    n = len(s)
    return -sum((c / n) * math.log2(c / n) for c in Counter(s).values())


# --------------------------------------------------------------------------- #
# Records, domains, episodes                                                    #
# --------------------------------------------------------------------------- #

# A benign corpus holds millions of Query objects and Episodes. __slots__ drops the per-instance dict
# (~50 B each); on Python < 3.10 the classes are simply ordinary dataclasses.
_DC = dict(slots=True) if sys.version_info >= (3, 10) else {}


@dataclass(**_DC)
class Query:
    """One DNS query observed on the wire."""
    ts: float
    src: str
    dst: str
    sport: int
    qname: str
    qtype: int
    payload: int = 0          # encoded payload characters (set by attribute_payload)
    tunnel: str = ""          # inferred tunnel suffix
    file: str = ""            # capture file it came from: episodes never span files


@dataclass(**_DC)
class Episode:
    eid: str
    tool: str
    src: str
    tunnel: str
    queries: list[Query] = field(default_factory=list)
    family: str = ""
    meta: dict = field(default_factory=dict)

    @property
    def t0(self) -> float: return self.queries[0].ts
    @property
    def t1(self) -> float: return self.queries[-1].ts
    @property
    def duration(self) -> float: return self.t1 - self.t0
    @property
    def asset(self) -> int: return sum(q.payload for q in self.queries)
    @property
    def n_queries(self) -> int: return len(self.queries)
    @property
    def n_ports(self) -> int: return len({q.sport for q in self.queries})


# Second-level labels under which registrations sit one level deeper (co.uk, com.au ...)
_SLD = {"co", "com", "net", "org", "ac", "gov", "edu", "or", "ne", "go", "mil", "nic"}


def registered_domain(name: str) -> str:
    """eTLD+1 heuristic. Exact for the tunnel domains that matter here; for benign
    traffic it only needs to group a client's queries by the domain they serve."""
    labels = name.rstrip(".").lower().split(".")
    if len(labels) <= 2:
        return ".".join(labels)
    if len(labels[-1]) == 2 and labels[-2] in _SLD:
        return ".".join(labels[-3:])
    return ".".join(labels[-2:])


def label_suffix(qnames: Iterable[str], min_labels: int = 2) -> str:
    """Longest common label-wise suffix shared by every qname, capped so that at
    least one label is always left of it (a tunnel query must carry a payload label;
    without the cap a run of identical names reports zero payload)."""
    parts = [q.rstrip(".").lower().split(".") for q in qnames if q]
    if not parts:
        return ""
    max_len = max(1, min(len(p) for p in parts) - 1)
    common: list[str] = []
    for i in range(1, max_len + 1):
        seg = parts[0][-i]
        if all(p[-i] == seg for p in parts):
            common.insert(0, seg)
        else:
            break
    return ".".join(common)


def attribute_payload(queries: list[Query], tunnel: str) -> None:
    """Set q.payload to the encoded payload characters carried left of `tunnel`."""
    suffix = "." + tunnel if tunnel else ""
    for q in queries:
        name = q.qname.rstrip(".").lower()
        if tunnel and name.endswith(suffix):
            q.payload = len(name) - len(suffix)
        elif tunnel and name == tunnel:
            q.payload = 0
        else:
            q.payload = len(name)
        q.tunnel = tunnel
        q.payload = max(0, q.payload)


def episode_encoding_cv(ep: Episode) -> float:
    """Coefficient of variation of per-query payload length. Descriptive only:
    variable chunk sizes are harmless to A5; per-query overhead is what matters,
    and `measure_floor` tests that directly."""
    lens = [q.payload for q in ep.queries[:-1] if q.payload > 0]
    if len(lens) < 2:
        return 0.0
    mu = sum(lens) / len(lens)
    if mu == 0:
        return 0.0
    var = sum((x - mu) ** 2 for x in lens) / (len(lens) - 1)
    return math.sqrt(var) / mu


def build_episodes(queries: list[Query], tool: str, gap_s: float,
                   min_queries: int = 20, min_payload: int = 512,
                   dominant_only: bool = True, family: str = "",
                   log: list | None = None) -> list[Episode]:
    """Episodes from one tool's queries.

    1. Scope to the capture file. Lab captures reuse client IPs across files, so
       grouping across files would merge independent tool runs into one episode.
    2. Within a file, group by (client, registered domain).
    3. dominant_only: keep only the registered domain carrying the most payload
       in each file. Every capture is one tool run against one tunnel domain;
       everything else in the file is background DNS and must not become an
       exfiltration episode. If two clients carry that domain (a capture that saw
       both the stub->resolver and resolver->authoritative legs) only the larger
       is kept, so one transfer is never counted twice.
    4. Infer the tunnel suffix, attribute payload, split on silence > gap_s.
    """
    by_file: dict[str, list[Query]] = defaultdict(list)
    for q in queries:
        by_file[q.file].append(q)

    episodes: list[Episode] = []
    for fname, fqs in by_file.items():
        groups: dict[tuple[str, str], list[Query]] = defaultdict(list)
        for q in fqs:
            groups[(q.src, registered_domain(q.qname))].append(q)

        # payload potential per group (characters left of the inferred suffix)
        scored = []
        for key, qs in groups.items():
            qs.sort(key=lambda x: x.ts)
            tun = label_suffix([q.qname for q in qs])
            attribute_payload(qs, tun)
            scored.append((sum(q.payload for q in qs), key, qs, tun))
        scored.sort(key=lambda r: -r[0])

        keep = scored
        if dominant_only and scored:
            top_dom = scored[0][1][1]
            keep = [scored[0]]
            same_dom = [r for r in scored[1:] if r[1][1] == top_dom]
            bg = [r for r in scored[1:] if r[1][1] != top_dom]
            if log is not None:
                tot = sum(r[0] for r in scored) or 1
                log.append({"file": fname, "tool": tool, "tunnel_domain": top_dom,
                            "kept_src": scored[0][1][0],
                            "n_queries_file": len(fqs), "n_groups": len(scored),
                            "n_queries_kept": len(scored[0][2]), "payload_kept": scored[0][0],
                            "payload_share_kept": scored[0][0] / tot,
                            "n_background_domains": len({r[1][1] for r in bg}),
                            "n_duplicate_vantage": len(same_dom),
                            "duplicate_vantage_max_share":
                                max([r[0] / max(1, scored[0][0]) for r in same_dom] or [0.0])})

        for _, (src, regdom), qs, tun in keep:
            run: list[Query] = []
            for q in qs:
                if run and (q.ts - run[-1].ts) > gap_s:
                    episodes.append(Episode("", tool, src, tun, run, family or tool,
                                            {"file": fname}))
                    run = []
                run.append(q)
            if run:
                episodes.append(Episode("", tool, src, tun, run, family or tool,
                                        {"file": fname}))

    kept = []
    for ep in episodes:
        if ep.n_queries >= min_queries and ep.asset >= min_payload and ep.duration > 0:
            ep.eid = f"{tool}|{ep.meta.get('file','')}|{ep.src}->{ep.tunnel}|{ep.t0:.3f}"
            kept.append(ep)
        elif log is not None:
            log.append({"file": ep.meta.get("file", ""), "tool": tool,
                        "rejected_run": True, "n_queries": ep.n_queries,
                        "asset": ep.asset, "duration_s": ep.duration})
    return kept


def benign_units(queries: list[Query], gap_s: float, min_queries: int = 1) -> list[Episode]:
    """Benign traffic cut by EXACTLY the malicious construction -- file, client,
    registered domain, silence -- so a benign unit is built the way a malicious
    one is, and the false-positive burden is measured on comparable units.

    Memory: a measurement run holds millions of these (a unit is cheap only if it is small), so
    ids are short ("benign|<file number>|<unit number>"), every unit of a capture shares ONE meta
    dict (the notebook writes the same per-file value into it), and each distinct domain is ONE
    string. Grouping is per capture file, which bounds the working dictionary to one file."""
    by_file: dict[str, list[Query]] = {}
    for q in queries:
        by_file.setdefault(q.file, []).append(q)
    dom_cache: dict[str, str] = {}
    out: list[Episode] = []
    for fid, (fname, fqs) in enumerate(by_file.items()):
        meta = {"file": fname}                                  # shared by every unit of this capture
        by_key: dict[tuple, list[Query]] = defaultdict(list)
        for q in fqs:
            rd = registered_domain(q.qname)
            by_key[(q.src, dom_cache.setdefault(rd, rd))].append(q)
        for (src, regdom), qs in by_key.items():
            qs.sort(key=lambda x: x.ts)
            tun = label_suffix([q.qname for q in qs])
            attribute_payload(qs, dom_cache.setdefault(tun, tun))
            run: list[Query] = []
            for q in qs:
                if run and (q.ts - run[-1].ts) > gap_s:
                    if len(run) >= min_queries:
                        out.append(Episode(f"benign|{fid}|{len(out)}", "benign", src, regdom, run, "benign", meta))
                    run = []
                run.append(q)
            if len(run) >= min_queries:
                out.append(Episode(f"benign|{fid}|{len(out)}", "benign", src, regdom, run, "benign", meta))
    return out


# --------------------------------------------------------------------------- #
# Record emission — the consumers                                               #
# --------------------------------------------------------------------------- #

def export_records(ts: Sequence[float], t_active: float, t_idle: float,
                   capture_end: float) -> list[tuple[int, int, float]]:
    """nfstream-style flow export over one flow key.

    Returns (i0, i1, emit_ts): packets ts[i0..i1] are summarised by a record that
    becomes available at emit_ts. (A2) is enforced structurally -- emit_ts is never
    earlier than the last packet the record covers.
    """
    out: list[tuple[int, int, float]] = []
    n = len(ts)
    i0 = 0
    while i0 < n:
        start = ts[i0]
        deadline = start + t_active
        i = i0
        while i + 1 < n:
            nxt = ts[i + 1]
            if nxt > deadline or (nxt - ts[i]) > t_idle:
                break
            i += 1
        last = ts[i]
        if i + 1 < n:
            emit = (last + t_idle) if (ts[i + 1] - last) > t_idle else deadline
        else:
            emit = min(last + t_idle, max(capture_end, last))
        out.append((i0, i, max(emit, last)))
        i0 = i + 1
    return out


def records_per_query(ep: Episode, **_) -> list[tuple[list[int], float]]:
    """Per-query consumer: the record IS the query, available as it arrives."""
    return [([i], q.ts) for i, q in enumerate(ep.queries)]


def records_flow(ep: Episode, t_active: float = 1800.0, t_idle: float = 15.0,
                 capture_end: float | None = None, **_) -> list[tuple[list[int], float]]:
    """Flow-record consumer: 5-tuple flows with active and idle timeouts."""
    end = capture_end if capture_end is not None else ep.t1
    by_flow: dict[tuple, list[int]] = defaultdict(list)
    for i, q in enumerate(ep.queries):
        by_flow[(q.src, q.dst, q.sport, 53, 17)].append(i)
    out: list[tuple[list[int], float]] = []
    for idxs in by_flow.values():
        idxs.sort(key=lambda i: ep.queries[i].ts)
        ts = [ep.queries[i].ts for i in idxs]
        for i0, i1, emit in export_records(ts, t_active, t_idle, end):
            out.append((idxs[i0:i1 + 1], emit))
    out.sort(key=lambda r: r[1])
    return out


def records_session(ep: Episode, window_s: float = 300.0, **_) -> list[tuple[list[int], float]]:
    """Session consumer: queries grouped by silence < window_s, one record each.
    Credited with the EARLIEST emission physically possible (its last packet), so
    every session figure is a LOWER BOUND on that family's true floor."""
    groups: list[list[int]] = []
    cur: list[int] = []
    for i, q in enumerate(ep.queries):
        if cur and (q.ts - ep.queries[cur[-1]].ts) > window_s:
            groups.append(cur)
            cur = []
        cur.append(i)
    if cur:
        groups.append(cur)
    return [(g, ep.queries[g[-1]].ts) for g in groups]


CONSUMERS = {"per-query": records_per_query, "flow": records_flow, "session": records_session}


# --------------------------------------------------------------------------- #
# Floors                                                                        #
# --------------------------------------------------------------------------- #

def timeline(ep: Episode) -> tuple[list[float], list[int]]:
    ts, cum, run = [], [], 0
    for q in ep.queries:
        run += q.payload
        ts.append(q.ts)
        cum.append(run)
    return ts, cum


def fraction_lost_by(ep: Episode, alert_ts: float, overhead: int = 0) -> float:
    """V(alert_ts) / V(total). `overhead` removes a fixed per-query cost from every
    query before accounting -- the A5 unit-sensitivity check."""
    total = sum(max(0, q.payload - overhead) for q in ep.queries)
    if total <= 0:
        return float("nan")
    lost = 0
    for q in ep.queries:
        if q.ts <= alert_ts + 1e-12:
            lost += max(0, q.payload - overhead)
        else:
            break
    return lost / total


def overhead_estimate(ep: Episode) -> int:
    """Assumed per-query protocol overhead for the A5 sensitivity check.

    The shortest queries a tunnel sends are polls and acks, so the 5th-percentile
    length is a natural estimate -- but in a fixed-length tunnel the shortest
    query IS a full data chunk, and taking it as overhead would erase the whole
    asset. So the estimate is capped at half the median length: "assume up to half
    of a typical query is overhead". Removing a CONSTANT per-query cost leaves the
    ratio V(t)/V(total) exactly unchanged unless query lengths vary over time, so
    any non-zero delta this produces is attributable to length variation alone.
    """
    lens = sorted(q.payload for q in ep.queries if q.payload > 0)
    if not lens:
        return 0
    p5 = lens[int(0.05 * (len(lens) - 1))]
    med = lens[len(lens) // 2]
    return int(min(p5, 0.5 * med))


def measure_floor(ep: Episode, consumer: str, **kw) -> dict:
    """The architectural floor for one consumer on one episode (oracle, A4)."""
    recs = CONSUMERS[consumer](ep, **kw)
    if not recs:
        return {}
    first_emit = min(r[1] for r in recs)
    t_eff = max(0.0, first_emit - ep.t0)
    d = ep.duration
    measured = fraction_lost_by(ep, first_emit)
    h = overhead_estimate(ep)
    alt = fraction_lost_by(ep, first_emit, overhead=h)
    closed = min(1.0, t_eff / d) if d > 0 else 1.0
    return {
        "episode": ep.eid, "tool": ep.tool, "family": ep.family, "consumer": consumer,
        "file": ep.meta.get("file", ""), "direction": ep.meta.get("direction", ""),
        "rtype": ep.meta.get("rtype", ""),
        "n_queries": ep.n_queries, "n_ports": ep.n_ports,
        "asset_units": ep.asset, "duration_s": d,
        "t_eff_s": t_eff, "n_records": len(recs),
        "floor_measured": measured, "floor_closed_form": closed,
        "excess_pp": (measured - closed) * 100.0,
        "overhead_est": h, "floor_overhead_removed": alt,
        "unit_delta_pp": (alt - measured) * 100.0 if alt == alt else float("nan"),
        "encoding_cv": episode_encoding_cv(ep),
    }


# --------------------------------------------------------------------------- #
# Operating point — the manuscript's a-priori rule (closes reviewer item M2)     #
# --------------------------------------------------------------------------- #

_TINY_F32 = 1.401298464324817e-45        # the smallest positive float32: "alert on any positive score"
_MAX_F32 = 3.4028234663852886e38         # the largest finite float32


def _next_up_f32(x: float) -> float:
    """The smallest float32 strictly above x, as a float. Detector scores come out of float32 models, and a
    threshold held in float64 can round back onto a tie when the comparison is made in float32."""
    if x != x:
        return x
    if x >= _MAX_F32:
        return math.inf
    if x < -_MAX_F32:
        return -_MAX_F32
    if 0.0 <= x < _TINY_F32:                           # zero, and values that would round to zero in float32
        return _TINY_F32
    f = struct.unpack("<f", struct.pack("<f", x))[0]
    if f > x:
        return f
    b = struct.unpack("<i", struct.pack("<f", f))[0] + (1 if f > 0 else -1)
    return struct.unpack("<f", struct.pack("<i", b))[0]


def max_alerts_within(n: int, budget: float) -> int:
    """Largest k whose upper 95% Wilson bound on k/n is inside the budget; -1 if even k = 0 is not."""
    if n <= 0 or wilson_upper(0, n) > budget:
        return -1
    lo, hi = 0, n
    while lo < hi:                                     # wilson_upper is increasing in k
        mid = (lo + hi + 1) // 2
        if wilson_upper(mid, n) <= budget:
            lo = mid
        else:
            hi = mid - 1
    return lo


def _budget_used(upper: float, budget: float) -> float:
    return upper / budget if budget > 0 else (0.0 if upper == 0 else math.inf)


# What set the threshold (the `limit` of pick_threshold's meta dict):
LIMITS = ("budget", "tie plateau", "zero scores", "all fit", "not certifiable", "over budget")


def pick_threshold(benign_scores: Sequence[float], budget: float,
                   grid: Sequence[float] | None = None) -> tuple[float, dict]:
    """Most aggressive threshold whose UPPER 95% Wilson bound on the training burden sits inside the budget.
    Headroom comes from the number of benign units actually observed. Fixed a priori; never inspects held-out data.

    Scores are probabilities in [0, 1] (NaN counts as the lowest score: a unit that can never alert). A unit alerts
    when its score is >= the threshold.

    v4.2: with no grid the answer is exact to the resolution of the scores. The budget allows at most k benign
    units to alert; the (k+1)-th highest score must stay silent, so the threshold is the smallest observed score
    above it (an observed value, so a comparison in any float width agrees with it; the top of the gap between two
    observed scores). Only when no observed score lies above it is the threshold made up: the next float32 above
    the pivot, or the smallest positive float32 when the pivot is zero. v4.1 searched a grid whose lowest point
    was 0.005, and in the 7 Oct run that point was already inside the budget in 54 of 54 folds: the rule never
    bound. Pass `grid` to get that search back (its unreachable case now returns inf, as the rule does).

    The meta dict says what set the threshold in `limit`, and how much of the budget the training units use in
    `budget_used` (the Wilson upper bound over the budget; above 1 means over):
      "budget"           k_allowed units alert and one more would break the budget: the budget bound exactly;
      "tie plateau"      a group of equal scores straddles the budget's edge and cannot be split, so fewer than
                         k_allowed units alert (an RF with 300 trees gives only 301 distinct scores);
      "zero scores"      the model scores more benign units zero (or NaN) than the budget lets alert; the
                         threshold is 'any positive score' and the model, not the budget, limits sensitivity;
      "all fit"          the budget admits alerting on every benign unit;
      "not certifiable"  fewer than 381 units at a 1% budget: even zero alerts has an upper bound over the
                         budget, so nothing alerts (threshold inf);
      "over budget"      the units that alert anyway exceed the budget: only a score of +inf, which no probability
                         is, can do that. It is reported rather than hidden.
    `budget_bound` is True for the first two, False for the next two and None for the last two."""
    if budget != budget:
        raise ValueError("budget is NaN")
    scores = list(benign_scores)                       # a generator is consumed once
    n_nan = sum(1 for x in scores if x != x)
    s = sorted(-math.inf if x != x else x for x in scores)    # a NaN compares False with every threshold: it never alerts
    n = len(s)
    if n == 0:
        return (math.inf, {"n_benign": 0, "reason": "no benign units", "budget_bound": None,
                           "limit": "not certifiable", "budget_used": math.nan})
    if grid is not None:
        positive = [g for g in grid if g > 0]
        lowest = min(positive) if positive else None   # the most sensitive setting that is not "alert on everything"
        for thr in sorted(grid):
            k = n - bisect.bisect_left(s, thr)
            upper = wilson_upper(k, n)
            if upper <= budget:
                return (thr, {"n_benign": n, "train_burden": k / n, "train_upper": upper, "threshold": thr,
                              "budget_bound": lowest is not None and thr > lowest,
                              "budget_used": _budget_used(upper, budget), "n_nan": n_nan})
        k = n - bisect.bisect_left(s, math.inf)
        return (math.inf, {"n_benign": n, "train_burden": k / n, "train_upper": wilson_upper(k, n),
                           "threshold": math.inf, "reason": "budget unreachable", "budget_bound": None,
                           "limit": "not certifiable", "budget_used": _budget_used(wilson_upper(k, n), budget),
                           "n_nan": n_nan})
    k_max = max_alerts_within(n, budget)
    if k_max < 0:                                      # too few benign units to certify even zero alerts: alert on nothing
        k = n - bisect.bisect_left(s, math.inf)        # (only a score of +inf could still alert)
        upper = wilson_upper(k, n)
        return (math.inf, {"n_benign": n, "train_burden": k / n, "train_upper": upper, "threshold": math.inf,
                           "reason": "budget unreachable", "budget_bound": None, "limit": "not certifiable",
                           "budget_used": _budget_used(upper, budget), "k_allowed": k_max, "n_nan": n_nan})
    if k_max >= n:
        thr, limit = 0.0, "all fit"                    # the budget admits alerting on everything
    else:
        pivot = s[n - k_max - 1]                       # the (k_max+1)-th highest score: it has to stay silent
        if pivot == 0.0 or pivot == -math.inf:         # (-inf: the NaN scores, which can never alert)
            thr, limit = _TINY_F32, "zero scores"      # the model's own zeros, not the budget, limit sensitivity
        else:
            j = bisect.bisect_right(s, pivot)
            thr = s[j] if j < n else _next_up_f32(pivot)
            limit = "budget"
    k = n - bisect.bisect_left(s, thr)
    if limit == "budget" and k < k_max:
        limit = "tie plateau"                          # equal scores at the edge: the budget cannot be spent in full
    upper = wilson_upper(k, n)
    if upper > budget:
        limit = "over budget"
    bound = {"budget": True, "tie plateau": True, "zero scores": False, "all fit": False}.get(limit)
    return (thr, {"n_benign": n, "train_burden": k / n, "train_upper": upper, "threshold": thr,
                  "budget_bound": bound, "limit": limit,
                  "budget_used": _budget_used(upper, budget), "k_allowed": k_max, "n_nan": n_nan})


def limit_summary(metas: Iterable[dict]) -> dict:
    """Counts of what set each fold's threshold (see `pick_threshold`), and how much of the budget was used where
    the budget could be certified. Used by the digest so that 'the budget was binding' is only said when it was."""
    metas = list(metas)
    out = {name: sum(1 for m in metas if m.get("limit") == name) for name in LIMITS}
    used = sorted(m["budget_used"] for m in metas
                  if m.get("limit") != "not certifiable" and m.get("budget_used") is not None
                  and m["budget_used"] == m["budget_used"])
    mid = len(used) // 2
    med = math.nan if not used else (used[mid] if len(used) % 2 else (used[mid - 1] + used[mid]) / 2)
    out.update(folds=len(metas), used_median=med,
               used_min=(used[0] if used else math.nan), used_max=(used[-1] if used else math.nan))
    return out


def alert_time(ep: Episode, consumer: str, score_of, threshold: float, **kw) -> float | None:
    recs = CONSUMERS[consumer](ep, **kw)
    fired = [emit for idxs, emit in recs if score_of(ep, idxs, emit) >= threshold]
    return min(fired) if fired else None


def measure_ebd(ep: Episode, consumer: str, score_of, threshold: float, **kw) -> dict:
    """EBD for a real detector, against the floor that binds it."""
    fl = measure_floor(ep, consumer, **kw)
    t = alert_time(ep, consumer, score_of, threshold, **kw)
    ebd = 1.0 if t is None else fraction_lost_by(ep, t)
    fl.update({"alerted": t is not None, "alert_ts": t, "ebd": ebd,
               "gap_pp": (ebd - fl["floor_measured"]) * 100.0,
               "at_floor": abs(ebd - fl["floor_measured"]) < 1e-9})
    return fl


# --------------------------------------------------------------------------- #
# ibHH-style streaming accumulator (outside the record-consuming family)        #
# --------------------------------------------------------------------------- #

def unique_info_trace(queries: Sequence[Query], window_s: float) -> list[tuple[float, int]]:
    """Exact count of UNIQUE subdomain characters sent to one registered domain
    within a sliding window -- the quantity ibHH estimates with a constant-memory
    sketch. Exact counting here, so this is ibHH-STYLE, not ibHH itself."""
    out: list[tuple[float, int]] = []
    win: deque = deque()
    cnt: Counter = Counter()
    info = 0
    for q in queries:
        rd = registered_domain(q.qname)
        name = q.qname.rstrip(".").lower()
        sub = name[: -len(rd) - 1] if name.endswith("." + rd) else ""
        while win and q.ts - win[0][0] > window_s:
            _, old = win.popleft()
            cnt[old] -= 1
            if cnt[old] == 0:
                info -= len(old)
                del cnt[old]
        if cnt[sub] == 0:
            info += len(sub)
        cnt[sub] += 1
        win.append((q.ts, sub))
        out.append((q.ts, info))
    return out


def ibhh_threshold(benign_streams: Sequence[Sequence[Query]], window_s: float,
                   budget: float) -> tuple[float, dict]:
    """Smallest information threshold whose upper Wilson bound on the fraction of
    TRAINING benign streams that ever reach it sits inside the budget."""
    maxima = sorted(max((v for _, v in unique_info_trace(s, window_s)), default=0)
                    for s in benign_streams)
    n = len(maxima)
    if n == 0:
        return (float("inf"), {"n_benign_streams": 0})
    for thr in sorted(set(maxima)) + [maxima[-1] + 1]:
        k = n - bisect.bisect_left(maxima, thr)
        if wilson_upper(k, n) <= budget:
            return (float(thr), {"n_benign_streams": n, "train_burden": k / n,
                                 "train_upper": wilson_upper(k, n)})
    return (float("inf"), {"n_benign_streams": n})


def ibhh_alert_time(ep: Episode, threshold: float, window_s: float) -> float | None:
    for ts, v in unique_info_trace(ep.queries, window_s):
        if v >= threshold:
            return ts
    return None


# --------------------------------------------------------------------------- #
# Runtime invariants                                                            #
# --------------------------------------------------------------------------- #

class InvariantViolation(AssertionError):
    pass


def check_invariants(ep: Episode, floors: dict[str, dict], strict: bool = True) -> list[str]:
    """HARD (raise): I1 reconciliation, I2 monotone timeline, I3 emission never
    precedes its traffic, I5 pipeline-position ordering, I6 EBD >= floor.
    SOFT (report): I4 measured floor vs the closed form min(1, T/D) -- a
    back-loaded transfer legitimately sits below it; asserting it would hide that."""
    notes: list[str] = []
    if sum(q.payload for q in ep.queries) != ep.asset:
        raise InvariantViolation(f"I1 reconciliation failed on {ep.eid}")
    ts, cum = timeline(ep)
    if any(b < a for a, b in zip(cum, cum[1:])) or any(b < a for a, b in zip(ts, ts[1:])):
        raise InvariantViolation(f"I2 timeline not monotone on {ep.eid}")
    for name, fl in floors.items():
        if not fl:
            continue
        if fl["t_eff_s"] < -1e-9:
            raise InvariantViolation(f"I3 record emitted before its traffic on {ep.eid}/{name}")
        if fl["excess_pp"] < -1e-6:
            notes.append(f"I4 {ep.eid}/{name}: measured floor {fl['excess_pp']:.2f} pp "
                         f"below the closed form (transfer ran behind the constant-rate diagonal)")
        if "ebd" in fl and fl["ebd"] + 1e-9 < fl["floor_measured"]:
            raise InvariantViolation(
                f"I6 EBD {fl['ebd']:.6f} < floor {fl['floor_measured']:.6f} on "
                f"{ep.eid}/{name} -- a detector cannot beat its own floor. "
                f"Proposition 1 is falsified, or there is a bug.")
    order = ["per-query", "flow", "session"]
    have = [floors[k]["floor_measured"] for k in order if floors.get(k)]
    for a, b in zip(have, have[1:]):
        if b + 1e-9 < a:
            msg = f"I5 pipeline-position ordering violated on {ep.eid}: {have}"
            if strict:
                raise InvariantViolation(msg)
            notes.append(msg)
    return notes

In [ ]:
from __future__ import annotations   # ===== pcap_io.py =====
"""PCAP/PCAPNG -> DNS query records.

Pure standard library. Kaggle notebooks frequently run with internet disabled,
so depending on dpkt/scapy being installable is a way for the run to die four
hours in. Nothing here needs them.

Link types read: Ethernet (VLAN, QinQ, PPPoE), Linux cooked capture v1 and v2
(what `tcpdump -i any` writes on old and new libpcap), BSD/OpenBSD loopback, raw IP
(101, 12, 14, 228, 229) and NFLOG. Any other link type is counted per file and
reported, never guessed: a capture the parser cannot read must show up as unreadable,
not as a capture that happens to contain no DNS.
"""

import gzip
import os
import socket
import struct
from collections import Counter
from typing import Iterator

# Query is defined in the engine cell above

# link types (tcpdump.org/linktypes.html)
DLT_NULL, DLT_EN10MB, DLT_RAW, DLT_LOOP, DLT_LINUX_SLL = 0, 1, 101, 108, 113
DLT_IPV4, DLT_IPV6, DLT_NFLOG, DLT_LINUX_SLL2 = 228, 229, 239, 276
DLT_RAW_ALT = {12, 14}
LINKTYPE_NAMES = {0: "BSD loopback", 1: "Ethernet", 12: "raw IP", 14: "raw IP",
                  101: "raw IP", 108: "OpenBSD loopback", 113: "Linux cooked",
                  228: "raw IPv4", 229: "raw IPv6", 239: "NFLOG", 276: "Linux cooked v2"}
SUPPORTED_LINKTYPES = set(LINKTYPE_NAMES)
_IP_ETHERTYPES = (0x0800, 0x86DD)


def linktype_name(lt: int) -> str:
    return LINKTYPE_NAMES.get(lt, f"UNSUPPORTED({lt})")


def _open(path: str):
    return gzip.open(path, "rb") if path.endswith(".gz") else open(path, "rb")


# --------------------------------------------------------------------------- #
# Container readers                                                             #
# --------------------------------------------------------------------------- #

def _iter_pcap(fh) -> Iterator[tuple[float, bytes, int]]:
    hdr = fh.read(24)
    if len(hdr) < 24:
        return
    magic = struct.unpack("<I", hdr[:4])[0]
    if magic in (0xA1B2C3D4, 0xA1B23C4D):
        endian, nano = "<", magic == 0xA1B23C4D
    elif magic in (0xD4C3B2A1, 0x4D3CB2A1):
        endian, nano = ">", magic == 0x4D3CB2A1
    else:
        return
    # low 16 bits are the link type; the high bits may carry FCS information
    linktype = struct.unpack(endian + "I", hdr[20:24])[0] & 0xFFFF
    div = 1e9 if nano else 1e6
    ph = struct.Struct(endian + "IIII")
    while True:
        rec = fh.read(16)
        if len(rec) < 16:
            return
        sec, frac, caplen, _ = ph.unpack(rec)
        data = fh.read(caplen)
        if len(data) < caplen:
            return
        yield (sec + frac / div, data, linktype)


def _iter_pcapng(fh) -> Iterator[tuple[float, bytes, int]]:
    """PCAPNG. Each Enhanced Packet Block names its interface; linktype and
    timestamp resolution are per interface, so they are tracked per interface
    rather than taken from whichever IDB happened to come last."""
    endian = "<"
    ifaces: list[tuple[int, float]] = []          # (linktype, ticks per second)
    last_ts = None
    while True:
        head = fh.read(8)
        if len(head) < 8:
            return
        btype = struct.unpack(endian + "I", head[:4])[0]
        if btype == 0x0A0D0D0A:                              # SHB: new section
            bom = fh.read(4)
            if len(bom) < 4:
                return
            endian = "<" if struct.unpack("<I", bom)[0] == 0x1A2B3C4D else ">"
            blen = struct.unpack(endian + "I", head[4:8])[0]
            fh.read(max(0, blen - 12))                       # 8 (head) + 4 (BOM) consumed
            ifaces = []                                      # interface ids reset per section
            continue
        blen = struct.unpack(endian + "I", head[4:8])[0]
        if blen < 12:
            return
        body = fh.read(blen - 12)
        fh.read(4)
        if btype == 0x00000001 and len(body) >= 8:           # IDB
            linktype = struct.unpack(endian + "H", body[:2])[0]
            tsres = 1e6
            opts, i = body[8:], 0
            while i + 4 <= len(opts):
                ocode, olen = struct.unpack(endian + "HH", opts[i:i + 4])
                val = opts[i + 4:i + 4 + olen]
                if ocode == 9 and val:                       # if_tsresol
                    r = val[0]
                    tsres = float(2 ** (r & 0x7F)) if r & 0x80 else float(10 ** r)
                if ocode == 0:
                    break
                i += 4 + olen + ((4 - olen % 4) % 4)
            ifaces.append((linktype, tsres))
        elif btype == 0x00000006 and len(body) >= 20:        # EPB
            iface, hi, lo, caplen = struct.unpack(endian + "IIII", body[:16])
            lt, res = ifaces[iface] if iface < len(ifaces) else (DLT_EN10MB, 1e6)
            last_ts = ((hi << 32) | lo) / res
            yield (last_ts, body[20:20 + caplen], lt)
        elif btype == 0x00000002 and len(body) >= 20:        # obsolete Packet Block
            iface, _, hi, lo, caplen = struct.unpack(endian + "HHIII", body[:16])
            lt, res = ifaces[iface] if iface < len(ifaces) else (DLT_EN10MB, 1e6)
            last_ts = ((hi << 32) | lo) / res
            yield (last_ts, body[20:20 + caplen], lt)
        elif btype == 0x00000003 and len(body) >= 4:         # Simple Packet Block: no timestamp
            lt = ifaces[0][0] if ifaces else DLT_EN10MB
            yield (None, body[4:], lt)                       # caller counts and skips it


def iter_packets(path: str) -> Iterator[tuple[float, bytes, int]]:
    with _open(path) as fh:
        magic = fh.read(4)
        fh.seek(0)
        if magic == b"\x0a\x0d\x0d\x0a":
            yield from _iter_pcapng(fh)
        else:
            yield from _iter_pcap(fh)


# --------------------------------------------------------------------------- #
# Layers                                                                        #
# --------------------------------------------------------------------------- #

def _nflog_ip(buf: bytes) -> tuple[bytes, int] | None:
    """NFLOG: 4-byte header, then TLVs (host byte order); the IP packet is NFULA_PAYLOAD (9)."""
    off = 4
    while off + 4 <= len(buf):
        tlen, ttype = struct.unpack("<HH", buf[off:off + 4])
        if tlen < 4:
            return None
        if ttype & 0x7FFF == 9:
            return (buf[off + 4:off + tlen], 0)
        off += (tlen + 3) & ~3
    return None


def _l3(buf: bytes, linktype: int) -> tuple[bytes, int] | None:
    """-> (buf, offset of the IP header), or None when the frame carries no IP.
    Unknown link types return None: the caller counts them instead of guessing."""
    if linktype == DLT_EN10MB:
        if len(buf) < 14:
            return None
        et = struct.unpack("!H", buf[12:14])[0]
        off = 14
        while et in (0x8100, 0x88A8, 0x9100) and len(buf) >= off + 4:   # VLAN / QinQ tags
            et = struct.unpack("!H", buf[off + 2:off + 4])[0]
            off += 4
        if et == 0x8864 and len(buf) >= off + 8:                       # PPPoE session
            ppp = struct.unpack("!H", buf[off + 6:off + 8])[0]
            return (buf, off + 8) if ppp in (0x0021, 0x0057) else None
        return (buf, off) if et in _IP_ETHERTYPES else None
    if linktype == DLT_LINUX_SLL:                    # cooked v1: 16 bytes, protocol at 14
        if len(buf) < 16:
            return None
        et = struct.unpack("!H", buf[14:16])[0]
        return (buf, 16) if et in _IP_ETHERTYPES else None
    if linktype == DLT_LINUX_SLL2:                   # cooked v2: 20 bytes, protocol at 0
        if len(buf) < 20:
            return None
        et = struct.unpack("!H", buf[0:2])[0]
        return (buf, 20) if et in _IP_ETHERTYPES else None
    if linktype in (DLT_NULL, DLT_LOOP):
        return (buf, 4) if len(buf) >= 4 else None
    if linktype in (DLT_RAW, DLT_IPV4, DLT_IPV6) or linktype in DLT_RAW_ALT:
        return (buf, 0)
    if linktype == DLT_NFLOG:
        return _nflog_ip(buf)
    return None


def _l4(buf: bytes, off: int):
    """-> (src, dst, proto, payload) or None."""
    if len(buf) < off + 1:
        return None
    ver = buf[off] >> 4
    if ver == 4:
        if len(buf) < off + 20:
            return None
        ihl = (buf[off] & 0x0F) * 4
        if ihl < 20 or len(buf) < off + ihl:
            return None
        proto = buf[off + 9]
        frag = struct.unpack("!H", buf[off + 6:off + 8])[0]
        if frag & 0x1FFF:                       # non-first fragment: no L4 header
            return None
        tot = struct.unpack("!H", buf[off + 2:off + 4])[0]
        end = off + tot if 0 < tot <= len(buf) - off else len(buf)
        return (buf[off + 12:off + 16], buf[off + 16:off + 20], proto, buf[off + ihl:end])
    if ver == 6:
        if len(buf) < off + 40:
            return None
        plen = struct.unpack("!H", buf[off + 4:off + 6])[0]
        end = off + 40 + plen if 0 < plen <= len(buf) - off - 40 else len(buf)
        nh, p = buf[off + 6], off + 40
        for _ in range(8):                                  # walk extension headers
            if nh in (0, 43, 60):                           # hop-by-hop, routing, dest opts
                if len(buf) < p + 2:
                    return None
                nh, p = buf[p], p + (buf[p + 1] + 1) * 8
            elif nh == 44:                                  # fragment header
                if len(buf) < p + 8:
                    return None
                if struct.unpack("!H", buf[p + 2:p + 4])[0] & 0xFFF8:
                    return None                             # non-first fragment
                nh, p = buf[p], p + 8
            elif nh == 51:                                  # authentication header
                if len(buf) < p + 2:
                    return None
                nh, p = buf[p], p + (buf[p + 1] + 2) * 4
            else:
                break
        if p > end:
            return None
        return (buf[off + 8:off + 24], buf[off + 24:off + 40], nh, buf[p:end])
    return None


MAX_NAME_HOPS = 128      # compression pointers followed in one name; real queries use none, so this only bounds garbage
MAX_NAME_OCTETS = 255    # RFC 1035 2.3.4: a name is at most 255 octets on the wire, length bytes and root included
HEADER_LEN = 12          # a compression pointer leads to a name, and no name starts inside the header


def _qname(msg: bytes, pos: int) -> tuple[str, int] | None:
    """Decode a DNS name. Returns (name, next_offset).

    Iterative since v4.2. v4.1 called itself once per compression pointer and restarted its label counter on
    every call, so a long chain of backward pointers ran into Python's recursion limit, and the per-capture
    handler then ended the WHOLE capture: four of the thirteen university captures of the 7 Oct run stopped
    that way (at 6.0, 21.8, 29.4 and 51.8 minutes). The names accepted are the same as before: only backward
    jumps, at most 8 labels before a jump, at most 128 labels in a stretch of the name. What is new is a cap on
    the number of jumps, which no real name comes near, and two rules of RFC 1035 that v4.1 did not enforce and
    no real packet breaks: a pointer cannot lead into the 12-byte header (v4.1 decoded header bytes as labels),
    and a name is at most 255 octets (v4.1 took longer ones). Both only ever turn garbage away."""
    labels, seen, hops, end, wire = [], 0, 0, None, 1      # wire: octets the name takes uncompressed (root byte counted)
    n_msg = len(msg)
    while True:
        if pos >= n_msg:
            return None
        n = msg[pos]
        if n == 0:
            if hops and seen == 0:              # the stretch after the last jump was the root alone (v4.1 appended "" for it)
                labels.append("")
            return (".".join(labels), pos + 1 if end is None else end)
        if n & 0xC0 == 0xC0:                    # compression pointer
            if pos + 2 > n_msg:
                return None
            ptr = ((n & 0x3F) << 8) | msg[pos + 1]
            if ptr >= pos or ptr < HEADER_LEN or seen > 8:     # only backward jumps, never into the header; bound the chase
                return None
            if end is None:
                end = pos + 2                   # the name ends at its first pointer, wherever the pointer leads
            hops += 1
            if hops > MAX_NAME_HOPS:
                return None
            pos, seen = ptr, 0
            continue
        pos += 1
        if pos + n > n_msg or n > 63:
            return None
        wire += n + 1
        if wire > MAX_NAME_OCTETS:
            return None
        labels.append(msg[pos:pos + n].decode("latin-1"))
        pos += n
        seen += 1
        if seen > 128:
            return None


def malformed_reason(msg: bytes) -> str:
    """Why parse_dns_question() rejected a port-53 payload. Diagnosis only: it is called once
    per rejected message, never on the hot path. (The 4 Oct smoke run had 35% of the port-53
    messages in one university capture rejected with no way to say why.)"""
    if len(msg) < 12:
        return "shorter than a DNS header"
    if struct.unpack("!H", msg[4:6])[0] < 1:
        return "question count is 0"
    pos, seen, wire = 12, 0, 1
    while True:
        if pos >= len(msg):
            return "name runs past the end of the message"
        n = msg[pos]
        if n == 0:
            return "question cut before QTYPE/QCLASS"
        if n & 0xC0 == 0xC0:
            return "compression pointer inside the question"
        if n & 0xC0:
            return "reserved label bits (0x40/0x80)"
        if pos + 1 + n > len(msg):
            return "label runs past the end of the message"
        pos += 1 + n
        seen += 1
        wire += n + 1
        if wire > MAX_NAME_OCTETS:
            return "name longer than 255 octets"
        if seen > 128:                                 # (cannot be reached while names are capped at 255 octets: a bound)
            return "more than 128 labels"


def parse_dns_question(msg: bytes):
    """-> (qname, qtype, is_query) or None."""
    if len(msg) < 12:
        return None
    flags, qd = struct.unpack("!HH", msg[2:6])
    if qd < 1:
        return None
    got = _qname(msg, 12)
    if got is None:
        return None
    name, p = got
    if p + 4 > len(msg):
        return None
    qtype = struct.unpack("!H", msg[p:p + 2])[0]
    return (name, qtype, (flags >> 15) == 0)


# --------------------------------------------------------------------------- #
# Public entry point                                                            #
# --------------------------------------------------------------------------- #

_NTOP_CACHE: dict[bytes, str] = {}


def _ntop(a: bytes) -> str:
    """Address text. Cached, so the few thousand distinct hosts in a capture are ONE string object
    each instead of one per query (a 6-million-query benign corpus holds ~130 B less per query)."""
    key = bytes(a)
    s = _NTOP_CACHE.get(key)
    if s is None:
        try:
            s = socket.inet_ntop(socket.AF_INET if len(key) == 4 else socket.AF_INET6, key)
        except Exception:
            s = key.hex()
        if len(_NTOP_CACHE) < 1_000_000:
            _NTOP_CACHE[key] = s
    return s


def parse_pcap(path: str, max_packets: int | None = None,
               queries_only: bool = True, file_tag: str | None = None
               ) -> tuple[list[Query], dict]:
    """Extract DNS queries from one capture.

    The returned stats carry the denominators that make a partially-read or
    skipped file visible in the results rather than silently absent.

    v4.2: a packet that raises is counted and skipped, and so is a message the decoder cannot take; neither ends
    the capture any more (v4.1 lost four university captures to one exception each). Only a failure of the
    container reader itself, a corrupt tail, ends a capture, and then `error` says so.

    DNS over TCP: a message normally follows its 2-byte length in the same segment (offset 2, as before). Some
    clients send the length on its own first. A bare 2-byte segment is remembered per direction together with the
    sequence number of the byte after it, and the next segment of that connection is taken for the message only if
    it starts at exactly that sequence number AND is exactly as long as the prefix announced AND parses as a DNS
    message with one question; its message is then read from offset 0 (`tcp_split_read`). Anything else falls back
    to offset 2 as in v4.1. The checks keep a stray 2-byte segment from turning the next ordinary `[length][message]`
    segment into an empty name (the v4.2 review's counter-example: a root name parses from offset 0 of almost any
    such buffer). `tcp_prefix_seen` counts the remembered prefixes and `tcp_prefix_dropped` those that were replaced,
    evicted or not followed by a matching segment. Not handled, as in v4.1: two prefixed messages in one segment
    (only the first is read) and a message split over three segments, which lose TCP queries, and retransmissions,
    which repeat one.
    `malformed_c2s` counts the rejected payloads sent TO port 53: the denominator that matters for queries is
    those plus the queries parsed, since a rejected payload from port 53 is a response or a stream fragment.
    """
    qs: list[Query] = []
    file_tag = file_tag if file_tag is not None else os.path.basename(path)
    st = {"file": file_tag, "path": path, "packets": 0, "port53": 0, "dns": 0,
          "udp53": 0, "malformed": 0, "malformed_c2s": 0, "tcp_split_read": 0,
          "tcp_prefix_seen": 0, "tcp_prefix_dropped": 0,
          "packet_errors": 0, "packet_error_samples": "",
          "t_first": None, "t_last": None,
          "truncated": False, "error": None, "linktypes": "", "unsupported_linktype": 0,
          "non_ip": 0, "no_timestamp": 0, "first_bytes": "",
          "malformed_reasons": "", "malformed_reason_counts": {}, "malformed_samples": {}}
    lts: Counter = Counter()
    bad_why: Counter = Counter()
    bad_samples: dict[str, list[str]] = {}
    split: dict = {}              # (src, dst, sport, dport) -> (length announced by a bare 2-byte TCP segment, seq after it)
    errs: list[str] = []
    try:
        for ts, buf, lt in iter_packets(path):
            st["packets"] += 1
            lts[lt] += 1
            if st["packets"] == 1:
                st["first_bytes"] = buf[:40].hex()
            if ts is None:                                    # pcapng Simple Packet Block
                st["no_timestamp"] += 1
                continue
            if st["t_first"] is None:
                st["t_first"] = ts
            st["t_last"] = ts
            if max_packets and st["packets"] > max_packets:
                st["truncated"] = True
                break
            try:
                if lt not in SUPPORTED_LINKTYPES:
                    st["unsupported_linktype"] += 1
                    continue
                l3 = _l3(buf, lt)
                if l3 is None:
                    st["non_ip"] += 1
                    continue
                got = _l4(*l3)
                if got is None:
                    continue
                src, dst, proto, pay = got
                cands = None
                if proto == 17:                               # UDP
                    if len(pay) < 8:
                        continue
                    sport, dport = struct.unpack("!HH", pay[:4])
                    msg = pay[8:]
                elif proto == 6:                              # TCP
                    if len(pay) < 20:
                        continue
                    sport, dport = struct.unpack("!HH", pay[:4])
                    if 53 not in (sport, dport):
                        continue
                    data = pay[(pay[12] >> 4) * 4:]
                    key = (src, dst, sport, dport)
                    seq = struct.unpack("!I", pay[4:8])[0]
                    if len(data) == 2:                        # a length prefix sent on its own
                        st["tcp_prefix_seen"] += 1
                        if len(split) >= 4096:                # a flood of bare prefixes must not grow without bound
                            st["tcp_prefix_dropped"] += len(split)
                            split.clear()
                        if key in split:
                            st["tcp_prefix_dropped"] += 1     # replaced before its message came
                        split[key] = ((data[0] << 8) | data[1], (seq + 2) & 0xFFFFFFFF)
                        continue
                    if len(data) < 3:                         # a bare ACK or a stub: leave a remembered prefix alone
                        continue
                    pending = split.pop(key, None)
                    msg = data[2:]
                    if pending is not None:
                        if pending == (len(data), seq):       # the prefix came alone: the message starts here
                            cands = (data, msg)
                        else:
                            st["tcp_prefix_dropped"] += 1     # the next segment was not the message it announced
                else:
                    continue
                if 53 not in (sport, dport) or not msg:
                    continue
                st["port53"] += 1
                q = None
                if cands:
                    q = parse_dns_question(cands[0])          # offset 0 first when the prefix arrived on its own ...
                    if q is not None and struct.unpack("!H", cands[0][4:6])[0] != 1:
                        q = None                              # (a query has one question; anything else is a mis-pairing)
                    if q is not None:
                        st["tcp_split_read"] += 1
                if q is None:
                    q = parse_dns_question(msg)               # ... otherwise after the prefix, as in v4.1
                if q is None:
                    st["malformed"] += 1
                    if dport == 53:
                        st["malformed_c2s"] += 1
                    shown = cands[0] if cands else msg
                    why = malformed_reason(shown)
                    bad_why[why] += 1
                    smp = bad_samples.setdefault(why, [])
                    if len(smp) < 3:                         # first bytes + enough context to recognise it
                        smp.append(f"{'udp' if proto == 17 else 'tcp'} {sport}->{dport} len={len(shown)} "
                                   f"{shown[:40].hex()}")
                    continue
                name, qtype, is_q = q
                if queries_only and not is_q:
                    continue
                st["dns"] += 1
                st["udp53"] += 1 if proto == 17 else 0
                qs.append(Query(ts=float(ts), src=_ntop(src), dst=_ntop(dst), sport=int(sport),
                                qname=name, qtype=int(qtype), file=file_tag))
            except MemoryError:
                raise
            except Exception as e:                            # this packet only; the capture goes on
                st["packet_errors"] += 1
                if len(errs) < 3:
                    errs.append(f"{type(e).__name__}: {str(e)[:60]} at packet {st['packets']}")
    except Exception as e:                                     # a corrupt tail must not
        st["error"] = f"{type(e).__name__}: {e}"               # lose the packets before it
    st["linktypes"] = ", ".join(f"{linktype_name(k)}: {v:,}" for k, v in lts.most_common())
    st["malformed_reasons"] = "; ".join(f"{k}: {v:,}" for k, v in bad_why.most_common())
    st["malformed_reason_counts"] = dict(bad_why)
    st["malformed_samples"] = bad_samples
    st["packet_error_samples"] = "; ".join(errs)
    if st["packets"] == 0 and st["error"] is None:
        st["error"] = "no packets read (not a pcap/pcapng file, or empty)"
    return qs, st

### Self-test

The engine proves itself on analytically known values **before** touching the corpus, including the three ways real lab captures could silently corrupt the episode set.

In [ ]:
# ======================= ENGINE SELF-TEST (must pass) =======================
import struct, socket

def _dns_q(txid, name, qtype=16):
    out = struct.pack("!HHHHHH", txid, 0x0100, 1, 0, 0, 0)
    for lab in name.split("."):
        out += bytes([len(lab)]) + lab.encode()
    return out + b"\x00" + struct.pack("!HH", qtype, 1)

def _ip_udp(src, dst, sport, payload):
    udp = struct.pack("!HHHH", sport, 53, 8 + len(payload), 0) + payload
    return struct.pack("!BBHHHBBH", 0x45, 0, 20 + len(udp), 0, 0, 64, 17, 0) + \
        socket.inet_aton(src) + socket.inet_aton(dst) + udp

def _frame(src, dst, sport, payload):
    return b"\x00\x01\x02\x03\x04\x05" + b"\x06\x07\x08\x09\x0a\x0b" + b"\x08\x00" + \
        _ip_udp(src, dst, sport, payload)

def _write(path, frames, linktype=1):
    with open(path, "wb") as fh:
        fh.write(struct.pack("<IHHiIII", 0xA1B2C3D4, 2, 4, 0, 0, 262144, linktype))
        for ts, f in frames:
            fh.write(struct.pack("<IIII", int(ts), int(round((ts % 1) * 1e6)), len(f), len(f)) + f)

def _relink(frames, linktype):
    """Re-wrap Ethernet test frames in another link layer."""
    wrap = {113: lambda ip: struct.pack("!HHH", 0, 1, 6) + b"\x00" * 8 + b"\x08\x00" + ip,
            276: lambda ip: b"\x08\x00" + b"\x00\x00" + struct.pack("!I", 2) +
                            struct.pack("!HBB", 1, 0, 6) + b"\x00" * 8 + ip,
            228: lambda ip: ip, 101: lambda ip: ip, 147: lambda ip: ip}[linktype]
    return [(ts, wrap(f[14:])) for ts, f in frames]

def _synth(n=120, rate=2.0, tunnel="t.exfil-lab.net", ppq=True, t0=1.7e9, src="10.1.2.3"):
    out = []
    for i in range(n):
        lab = (f"{i:05d}" + "abcdefghijklmnopqrstuvwxyz234567" * 3)[:45]
        out.append((t0 + i / rate, _frame(src, "9.9.9.9", (40000 + i) if ppq else 40000,
                                          _dns_q(1000 + i, f"{lab}.{tunnel}"))))
    return out

_fails = []
def _ck(name, cond, d=""):
    print(f"  {'PASS' if cond else 'FAIL'}  {name}{('  — ' + d) if d else ''}")
    if not cond: _fails.append(name)

_write("/tmp/_st.pcap", _synth())
_qs, _st = parse_pcap("/tmp/_st.pcap")
_ck("parser recovers all 120 queries", len(_qs) == 120, f"{len(_qs)}")
for _lt, _nm in ((113, "Linux cooked v1"), (276, "Linux cooked v2 (tcpdump -i any)"), (228, "raw IPv4")):
    _write("/tmp/_lt.pcap", _relink(_synth(), _lt), linktype=_lt)
    _q, _s = parse_pcap("/tmp/_lt.pcap")
    _ck(f"link type {_lt} ({_nm}) decodes", len(_q) == 120, f"{len(_q)} queries, {_s['linktypes']}")
_write("/tmp/_lt.pcap", _relink(_synth(), 147), linktype=147)
_q, _s = parse_pcap("/tmp/_lt.pcap")
_ck("an unknown link type is reported, never guessed",
    not _q and _s["unsupported_linktype"] == 120 and "UNSUPPORTED(147)" in _s["linktypes"])
_ep = build_episodes(_qs, "selftest", gap_s=60.0)[0]
_ck("tunnel domain inferred without a tool signature", _ep.tunnel == "t.exfil-lab.net", _ep.tunnel)
_ck("asset reconciles exactly (I1)", _ep.asset == 120 * 45, str(_ep.asset))
_fl = {c: measure_floor(_ep, c, t_active=1800.0, t_idle=15.0, window_s=300.0)
       for c in ("per-query", "flow", "session")}
_ck("per-query floor == 1/120", abs(_fl["per-query"]["floor_measured"] - 1/120) < 1e-9)
_ck("flow floor == 31/120 at a 15 s idle timeout", abs(_fl["flow"]["floor_measured"] - 31/120) < 1e-9)
_ck("session floor == 100%", _fl["session"]["floor_measured"] == 1.0)
_ck("pipeline-position ordering holds (I5)", check_invariants(_ep, _fl) == [])
_ck("fixed-length tunnel: payload unit is exact (A5)", abs(_fl["flow"]["unit_delta_pp"]) < 1e-9)

# the three real-capture hazards
_a = parse_pcap("/tmp/_st.pcap", file_tag="a_up.pcapng")[0]
_write("/tmp/_st2.pcap", _synth(t0=1.7e9 + 20))           # same client, overlapping time
_b = parse_pcap("/tmp/_st2.pcap", file_tag="b_up.pcapng")[0]
_ck("client IP reused across files does NOT merge runs",
    len(build_episodes(_a + _b, "t", gap_s=300.0)) == 2)
_bg = [Query(ts=1.7e9 + i * .4, src="10.1.2.3", dst="9.9.9.9", sport=50000 + i,
             qname=f"x{i % 57}.update.vendor-cdn.com", qtype=1, file="a_up.pcapng")
       for i in range(300)]
_e = build_episodes(_a + _bg, "t", gap_s=300.0)
_ck("background DNS inside a tool capture is not an episode",
    len(_e) == 1 and _e[0].tunnel == "t.exfil-lab.net")
_write("/tmp/_st3.pcap", _synth(src="10.1.2.53", t0=1.7e9 + .02))
_leg2 = parse_pcap("/tmp/_st3.pcap", file_tag="a_up.pcapng")[0]
_ck("a two-leg capture counts the transfer once", len(build_episodes(_a + _leg2, "t", gap_s=300.0)) == 1)

_write("/tmp/_sp.pcap", _synth(ppq=False))
_ep2 = build_episodes(parse_pcap("/tmp/_sp.pcap")[0], "selftest", gap_s=60.0)[0]
_ck("single-port tunnel is a TOTAL LOSS for a flow consumer",
    measure_floor(_ep2, "flow", t_active=1800.0, t_idle=15.0)["floor_measured"] == 1.0)
_m = measure_ebd(_ep, "flow", lambda e, i, t: 1.0, 0.5, t_active=1800.0, t_idle=15.0)
_ck("an oracle sits exactly ON its floor (A4)", _m["at_floor"])
try:
    _bad = dict(_m); _bad["ebd"] = _bad["floor_measured"] - 0.1
    check_invariants(_ep, {"flow": _bad}); _ck("I6 raises when EBD < floor", False)
except InvariantViolation:
    _ck("I6 raises when EBD < floor", True)
_thr, _meta = pick_threshold([0.0] * 990 + [0.6] * 10, budget=0.01)
_ck("Wilson-upper operating point refuses the boundary threshold",
    _thr > 0.6 and _meta["train_upper"] <= 0.01 and _meta["budget_bound"], f"thr={_thr}")
_rng = random.Random(7)
_sc = [_rng.random() ** 8 for _ in range(20_000)]                 # skewed, no ties
_thr, _meta = pick_threshold(_sc, budget=0.01)
_k = sum(1 for x in _sc if x >= _thr)
_ck("exact rule: the threshold is an observed score", _thr in set(_sc), f"{_thr:.3e}")
_ck("exact rule: the burden is inside the budget and one more alert would break it",
    wilson(_k, len(_sc))[1] <= 0.01 < wilson(_k + 1, len(_sc))[1], f"{_k} of {len(_sc):,} alert")
_ck("exact rule: reports that the budget bound", _meta["budget_bound"] and _meta["k_allowed"] == _k)
_thr, _meta = pick_threshold([0.0] * 5000, budget=0.01)
_ck("exact rule: a model that scores every benign unit zero is not budget-bound",
    (not _meta["budget_bound"]) and 0.0 < _thr < 1e-30, f"thr={_thr}")
_thr, _meta = pick_threshold([0.99] * 50, budget=0.01)
_ck("exact rule: 50 benign units cannot certify the budget, so nothing alerts (float32 included)",
    _thr == float("inf") and not (np.float32(1.0) >= _thr))
_s32 = [np.float32(x) for x in np.random.default_rng(3).beta(0.05, 5, 50_000)]
_thr, _meta = pick_threshold(_s32, budget=0.01)
_a32 = sum(1 for x in _s32 if x >= _thr); _a64 = sum(1 for x in _s32 if float(x) >= float(_thr))
_ck("exact rule: float32 scores and a float64 threshold agree on who alerts",
    _a32 == _a64 == round(_meta["train_burden"] * len(_s32)), f"{_a32} / {_a64}")
_thr, _meta = pick_threshold([0.1] * 16800 + [0.7] * 3000 + [0.9] * 200, budget=0.01)
_ck("exact rule: equal scores at the edge of the budget are reported as a tie plateau, not as 'the budget was binding'",
    _meta["limit"] == "tie plateau" and _meta["train_burden"] == 0.0 and _meta["budget_used"] < 0.03, f"{_meta['limit']}, {_meta['budget_used']:.3f} used")
_thr, _meta = pick_threshold([0.99] * 50, budget=0.01)
_ck("exact rule: too few benign units to certify the budget is reported as such", _meta["limit"] == "not certifiable" and _meta["budget_bound"] is None)
_fm, _flo, _fhi = family_bootstrap({"a": [1.0, 1.0], "b": [0.0], "c": [0.0, 0.0, 0.0, 0.0]}, n_boot=500)
_ck("family-weighted mean gives every family the same weight", abs(_fm - 1 / 3) < 1e-12, f"{_fm:.4f}")
_tm, _tlo, _thi = family_t_interval({"a": [1.0], "b": [2.0], "c": [3.0], "d": [4.0]})
_ck("Student-t interval over family means (t(0.975, 3) = 3.1824)", abs(_tm - 2.5) < 1e-12 and abs(_thi - _tm - 3.182446305 * (5 / 3) ** .5 / 2) < 1e-6,
    f"[{_tlo:.4f}, {_thi:.4f}]")
_cl, _ch = clopper_pearson(3, 8)
_ck("exact (Clopper-Pearson) interval for 3 of 8 families", abs(_cl - 0.08523) < 1e-4 and abs(_ch - 0.75514) < 1e-4, f"[{_cl:.4f}, {_ch:.4f}]")
_tr = unique_info_trace([Query(ts=float(i), src="a", dst="b", sport=1, qname="www.example.com",
                               qtype=1) for i in range(20)], window_s=60)
_ck("ibHH-style statistic counts UNIQUE subdomain characters", _tr[-1][1] == 3)

# ---- v4.2: the parser hazards the 7 Oct run exposed ---------------------------------------------------------
# A name whose pointer leads back to its own start (v4.1 recursed until Python gave up, and the per-capture
# handler then ended the WHOLE capture: four university captures were cut that way).
_loop = struct.pack("!HHHHHH", 1, 0x0100, 1, 0, 0, 0) + b"\x01a\xc0\x0c" + b"\x00\x10\x00\x01"
try:
    _ck("a looping compression pointer is rejected, not fatal (v4.1: RecursionError)",
        parse_dns_question(_loop) is None)
except RecursionError:
    _ck("a looping compression pointer is rejected, not fatal (v4.1: RecursionError)", False)
_good = _synth(n=40, t0=1.7e9)
_write("/tmp/_lp.pcap", _good[:20] + [(1.7e9 + 20.5, _frame("10.1.2.3", "9.9.9.9", 41000, _loop))] + _good[20:])
_q, _s = parse_pcap("/tmp/_lp.pcap")
_ck("one hostile packet costs a capture nothing else", len(_q) == 40 and _s["error"] is None and _s["malformed"] == 1,
    f"{len(_q)} queries, malformed {_s['malformed']}, error {_s['error']}")
_orig_pdq = parse_dns_question
def _boom(msg):
    if msg[12:16] == b"\x01a\xc0\x0c":
        raise ValueError("synthetic failure")
    return _orig_pdq(msg)
parse_dns_question = _boom
try:
    _q, _s = parse_pcap("/tmp/_lp.pcap")
finally:
    parse_dns_question = _orig_pdq
_ck("a packet that raises is counted and skipped; the capture goes on",
    len(_q) == 40 and _s["packet_errors"] == 1 and _s["error"] is None and "synthetic failure" in _s["packet_error_samples"],
    f"{len(_q)} queries, packet_errors {_s['packet_errors']}")

def _tframe(src, dst, sport, payload, dport=53, seq=1):          # an Ethernet/IPv4/TCP frame
    tcp = struct.pack("!HHIIBBHHH", sport, dport, seq, 1, 0x50, 0x18, 65535, 0, 0) + payload
    ip = struct.pack("!BBHHHBBH", 0x45, 0, 20 + len(tcp), 0, 0, 64, 6, 0) + socket.inet_aton(src) + socket.inet_aton(dst) + tcp
    return b"\x00\x01\x02\x03\x04\x05" + b"\x06\x07\x08\x09\x0a\x0b" + b"\x08\x00" + ip
_m = _dns_q(77, "abc.t.exfil-lab.net")
_pre = struct.pack("!H", len(_m))
_write("/tmp/_tcp.pcap", [(1.7e9, _tframe("10.1.2.3", "9.9.9.9", 50000, _pre + _m)),      # length and message together
                          (1.7e9 + 1.0, _tframe("10.1.2.3", "9.9.9.9", 50001, _pre, seq=100)),     # length alone ...
                          (1.7e9 + 1.001, _tframe("10.1.2.3", "9.9.9.9", 50001, _m, seq=102))])    # ... then the message
_q, _s = parse_pcap("/tmp/_tcp.pcap")
_ck("DNS over TCP: a length prefix sent in a segment of its own is recovered",
    len(_q) == 2 and _s["tcp_split_read"] == 1 and all(x.qname == "abc.t.exfil-lab.net" for x in _q) and _s["malformed"] == 0,
    f"{len(_q)} queries, split reads {_s['tcp_split_read']}, malformed {_s['malformed']}")
# a stray 2-byte segment must not turn the next ordinary [length][message] segment into an empty name (v4.2 review)
_ord = _pre + _m
_write("/tmp/_tcp2.pcap", [(1.7e9, _tframe("10.1.2.3", "9.9.9.9", 50002, struct.pack("!H", len(_ord)), seq=500)),
                           (1.7e9 + .01, _tframe("10.1.2.3", "9.9.9.9", 50002, _ord, seq=9000)),
                           (1.7e9 + 1.0, _tframe("10.1.2.3", "9.9.9.9", 50003, struct.pack("!H", len(_ord)), seq=500)),
                           (1.7e9 + 1.01, _tframe("10.1.2.3", "9.9.9.9", 50003, _ord, seq=502))])
_q, _s = parse_pcap("/tmp/_tcp2.pcap")
_ck("DNS over TCP: a stray 2-byte segment does not turn the next ordinary segment into an empty name",
    [x.qname for x in _q] == ["abc.t.exfil-lab.net"] * 2 and _s["tcp_split_read"] == 0,
    f"{[x.qname for x in _q]}, split reads {_s['tcp_split_read']}")

assert not _fails, f"ENGINE SELF-TEST FAILED: {_fails}"
print("\n  engine self-test: all passed")

### Corpus discovery

The layout is read from the paths at runtime and printed. Check it before the long run: every malicious file needs a tool and a direction, and the `_down` captures must show as excluded.

In [ ]:
# ============================ CORPUS DISCOVERY ============================
PCAP_EXT = (".pcap", ".pcapng", ".cap", ".pcap.gz", ".pcapng.gz")
RTYPES = {"a", "aaaa", "txt", "cname", "null", "private", "key", "mix", "mx", "srv"}

def describe(path):
    rel = os.path.relpath(path, DATA_ROOT)
    parts = rel.split(os.sep)
    low = [p.lower() for p in parts]
    stem = re.sub(r"\.(pcapng|pcap|cap)(\.gz)?$", "", low[-1])
    toks = re.split(r"[_\-.]+", stem)
    if "malicious" in low[:-1]:
        i = low.index("malicious")
        tool = low[i + 1] if i + 1 < len(low) - 1 else (toks[0] if toks else "unknown")
        direction = "up" if "up" in toks else "down" if "down" in toks else "unspecified"
        rtype = next((t for t in toks if t in RTYPES), "")
        return {"kind": "malicious", "tool": tool, "family": tool.replace("_modified", ""),
                "direction": direction, "rtype": rtype, "rel": rel, "path": path}
    if "benign" in low[:-1]:
        i = low.index("benign")
        src = low[i + 1] if i + 1 < len(low) - 1 else "benign"
        return {"kind": "benign", "source": src, "rel": rel, "path": path}
    return {"kind": "unknown", "rel": rel, "path": path}

ALL, _seen = [], set()
for dirpath, dirnames, files in os.walk(DATA_ROOT, followlinks=True):   # mounts may be symlinks
    real = os.path.realpath(dirpath)
    if real in _seen:                      # a second path to the same mount, or a link loop
        dirnames[:] = []
        continue
    _seen.add(real)
    for f in files:
        if f.lower().endswith(PCAP_EXT):
            ALL.append(describe(os.path.join(dirpath, f)))
ALL.sort(key=lambda d: d["rel"])
print(f"capture files discovered: {len(ALL)}")

mal = [d for d in ALL if d["kind"] == "malicious"]
ben = [d for d in ALL if d["kind"] == "benign"]
unk = [d for d in ALL if d["kind"] == "unknown"]

print(f"\n{'family':18s} {'tool':24s} {'up':>4s} {'down':>5s} {'unspec':>7s}  record types")
print("-" * 82)
for fam in sorted({d["family"] for d in mal}):
    for tool in sorted({d["tool"] for d in mal if d["family"] == fam}):
        ds = [d for d in mal if d["tool"] == tool]
        c = Counter(d["direction"] for d in ds)
        print(f"{fam:18s} {tool:24s} {c['up']:4d} {c['down']:5d} {c['unspecified']:7d}  "
              f"{','.join(sorted({d['rtype'] or '-' for d in ds}))}")
print(f"\n{'benign source':28s} {'files':>6s} {'GB':>8s}")
for s in sorted({d["source"] for d in ben}):
    ds = [d for d in ben if d["source"] == s]
    print(f"{s:28s} {len(ds):6d} {sum(os.path.getsize(d['path']) for d in ds)/1e9:8.2f}")
if unk:
    print(f"\n  {len(unk)} captures outside malicious/ and benign/ — ignored:",
          [d['rel'] for d in unk[:4]])

def spread(items, k):
    """Deterministic spread across a sorted list, so a subset covers every day."""
    if len(items) <= k: return list(items)
    step = len(items) / k
    return [items[int(i * step)] for i in range(k)]

if not mal:
    _att = sorted(os.listdir(DATA_ROOT)) if os.path.isdir(DATA_ROOT) else []
    raise RuntimeError(
        f"No malicious captures under {DATA_ROOT}. "
        + (f"Attached here: {_att}. " if _att else "Nothing is attached to this notebook. ")
        + "Attach the corpus: Add Input -> Datasets -> search 'daumel dns-tunneling-dataset' "
          "('DNS exfiltration dataset') -> Add, then Run All. Importing an .ipynb into Kaggle "
          "does not bring its inputs with it.")
if not ben:
    print("\n  WARNING: no benign captures found -- floors will run, attainment cannot.")

USE_MAL = [d for d in mal if d["direction"] in DIRECTIONS_INCLUDED]
if not USE_MAL:
    raise RuntimeError(f"{len(mal)} malicious captures found, but none has a direction in "
                       f"{DIRECTIONS_INCLUDED}. Check the table above and DIRECTIONS_INCLUDED.")
if SMOKE:
    USE_MAL = [sorted([d for d in USE_MAL if d["family"] == f], key=lambda d: d["direction"] != "up")[0]
               for f in sorted({d["family"] for d in USE_MAL})]
EXCLUDED_DOWN = [d for d in mal if d["direction"] not in DIRECTIONS_INCLUDED]
top = [d for d in ben if d["source"] == "top_1_million_domains"]
uni = [d for d in ben if d["source"] == "university_traffic"]
other_ben = [d for d in ben if d["source"] not in ("top_1_million_domains", "university_traffic")]
USE_BEN = (top[:2] if SMOKE else top) + spread(uni, N_UNIVERSITY_FILES) + other_ben
print(f"\nSELECTED  malicious: {len(USE_MAL)} captures   (excluded as not exfiltration: "
      f"{len(EXCLUDED_DOWN)} '_down')")
print(f"          benign:    {len(USE_BEN)} captures  "
      f"(top-1M {sum(d in top for d in USE_BEN)}, university {sum(d in uni for d in USE_BEN)}"
      f" of {len(uni)})")

In [ ]:
# ============================ PARSE CAPTURES ============================
_require("USE_MAL", "USE_BEN", "uni")
t0 = time.time()
mal_queries = defaultdict(list)          # (tool, family) -> queries
ben_queries = []
PARSE = []

def _parse(d, cap):
    qs, st = parse_pcap(d["path"], max_packets=cap, file_tag=d["rel"])
    st.update({k: d.get(k, "") for k in ("kind", "tool", "family", "direction", "rtype", "source")})
    st["seconds"] = (st["t_last"] - st["t_first"]) if st["t_first"] is not None else 0.0
    PARSE.append(st)
    return qs

for i, d in enumerate(USE_MAL):
    mal_queries[(d["tool"], d["family"])].extend(_parse(d, None))   # never truncated
    print(f"\r  malicious {i+1:3d}/{len(USE_MAL)}  {time.time()-t0:6.0f}s", end="", flush=True)
print()

# Benign captures, with a memory plan. After the first university capture is in, its measured cost says how many
# more fit in this container's share (MEM_FRAC_PARSE of the limit). If they all fit, nothing changes. If not,
# the plan is cut to an evenly spaced subset of the discovered captures, and a backstop stops reading if the
# corpus outgrows its share anyway. Either way the digest says how many were read and why.
_uni_ids = {id(x) for x in uni}
_lim, _lim_how, _lim_raw = _mem_limit_info()
print(f"memory limit used for the plan: {_lim / 1e9:.1f} GB  [{_lim_how}]" if _lim else f"memory limit unknown  [{_lim_how}]")
MEM_GUARD = {"limit_gb": round(_lim / 1e9, 1) if _lim else None, "limit_source": _lim_how, "limit_raw": _lim_raw,
             "budget_gb": round(_lim * MEM_FRAC_PARSE / 1e9, 1) if _lim else None,
             "university_discovered": len(uni), "university_planned": sum(id(x) in _uni_ids for x in USE_BEN),
             "university_read": None, "capture_cost_gb": None, "action": "none"}
memory_mark("malicious parsed")
_sel, _read, _pos, _planned, _stop_uni = list(USE_BEN), [], 0, False, False
while _pos < len(_sel):
    d = _sel[_pos]; _pos += 1
    is_uni = id(d) in _uni_ids
    if (is_uni and _lim and not _stop_uni and sum(id(x) in _uni_ids for x in _read) >= MIN_UNI_FILES
            and (_mem_used_bytes() or 0) > _lim * MEM_FRAC_PARSE):
        _stop_uni = True                                    # backstop: the estimate below was too hopeful
        MEM_GUARD["action"] = "stopped reading"
    if is_uni and _stop_uni:
        continue
    _u0 = _mem_used_bytes() if (is_uni and not _planned) else None
    ben_queries.extend(_parse(d, MAX_PACKETS_BENIGN)); _read.append(d)
    if is_uni and not _planned:
        _planned = True
        _u1 = _mem_used_bytes(); _left = sum(id(x) in _uni_ids for x in _sel[_pos:])
        if _lim and _u0 and _u1 and _left:
            _cost = max(1.0, _u1 - _u0)
            MEM_GUARD["capture_cost_gb"] = round(_cost / 1e9, 3)
            _afford = int((_lim * MEM_FRAC_PARSE - _u1) // (1.5 * _cost))
            _k = min(MEM_GUARD["university_planned"], 1 + max(MIN_UNI_FILES - 1, _afford))   # never MORE than was planned
            if _k < MEM_GUARD["university_planned"]:
                _new = spread(uni, _k)
                _sel = (_sel[:_pos] + [x for x in _new if x is not d]
                        + [x for x in _sel[_pos:] if id(x) not in _uni_ids])
                MEM_GUARD["action"] = "reduced the plan"
    gb = sum(os.path.getsize(x["path"]) for x in _read) / 1e9
    print(f"\r  benign    {len(_read):3d}/{len(_sel)}  {gb:6.1f} GB  {len(ben_queries):10,d} queries  "
          f"{time.time()-t0:6.0f}s", end="", flush=True)
print()
USE_BEN[:] = _read                                           # everything downstream sees what was actually read
MEM_GUARD["university_read"] = sum(id(x) in _uni_ids for x in _read)
memory_mark("benign parsed")
if MEM_GUARD["action"] != "none":
    print(f"\n  MEMORY GUARD ({MEM_GUARD['action']}): this container allows {MEM_GUARD['limit_gb']} GB "
          f"[{MEM_GUARD['limit_source']}] and the parsed "
          f"corpus may use {MEM_GUARD['budget_gb']} GB of it (one university capture costs about "
          f"{MEM_GUARD['capture_cost_gb']} GB). Read {MEM_GUARD['university_read']} of "
          f"{MEM_GUARD['university_discovered']} university captures (planned {MEM_GUARD['university_planned']}). "
          f"The subset is evenly spaced over the discovered captures, so it samples the whole collection rather than "
          f"one stretch of it.")

print(f"\n{'tool':24s} {'files':>5s} {'packets':>12s} {'port-53':>10s} {'DNS queries':>12s} "
      f"{'malformed':>10s}  link type")
print("-" * 104)
for tool in sorted({s["tool"] for s in PARSE if s["kind"] == "malicious"}):
    ss = [s for s in PARSE if s["tool"] == tool]
    p53 = sum(s["port53"] for s in ss); bad = sum(s["malformed"] for s in ss)
    nq = sum(s["dns"] for s in ss)
    flag = ("   <-- NO DNS QUERIES" if nq == 0 else
            "   <-- check" if p53 and bad / p53 > 0.05 else "")
    lts = sorted({lt.split(":")[0] for s in ss for lt in s["linktypes"].split(", ") if lt})
    print(f"{tool:24s} {len(ss):5d} {sum(s['packets'] for s in ss):12,d} {p53:10,d} "
          f"{nq:12,d} {bad/max(1,p53)*100:9.2f}%  {', '.join(lts)}{flag}")
_bl = sorted({lt.split(":")[0] for s in PARSE if s["kind"] == "benign"
              for lt in s["linktypes"].split(", ") if lt})
print(f"{'(benign)':24s} {sum(s['kind'] == 'benign' for s in PARSE):5d} "
      f"{sum(s['packets'] for s in PARSE if s['kind'] == 'benign'):12,d} "
      f"{sum(s['port53'] for s in PARSE if s['kind'] == 'benign'):10,d} {len(ben_queries):12,d} "
      f"{'':10s}  {', '.join(_bl)}")

# The benign side is where realism rests, so it gets its own account -- including WHY port-53
# messages were rejected. (The 4 Oct smoke run rejected 35% of one university capture's port-53
# messages and could not say why.)
print(f"\n{'benign source':24s} {'files':>5s} {'packets':>12s} {'port-53':>10s} {'DNS queries':>12s} "
      f"{'malformed':>10s} {'of queries':>11s}  most common reasons")
for src in sorted({s["source"] for s in PARSE if s["kind"] == "benign"}):
    ss = [s for s in PARSE if s["kind"] == "benign" and s["source"] == src]
    p53 = sum(s["port53"] for s in ss); bad = sum(s["malformed"] for s in ss)
    nq_s = sum(s["dns"] for s in ss); bad_q = sum(s["malformed_c2s"] for s in ss)
    why = Counter()
    for s in ss:
        why.update(s["malformed_reason_counts"])
    print(f"{src:24s} {len(ss):5d} {sum(s['packets'] for s in ss):12,d} {p53:10,d} "
          f"{nq_s:12,d} {bad/max(1,p53)*100:9.2f}% {bad_q/max(1, bad_q + nq_s)*100:10.2f}%  "
          + "; ".join(f"{k} {v:,}" for k, v in why.most_common(3)))
    if bad / max(1, p53) > 0.05:
        print(f"    {bad/max(1,p53)*100:.0f}% of this source's port-53 messages are excluded from the "
              f"benign units ({bad_q/max(1, bad_q + nq_s)*100:.1f}% of those sent TO port 53, which is where queries are). "
              f"Samples are in dns_malformed_samples.csv; send that file back.")
_nt = sum(s["truncated"] for s in PARSE if s["kind"] == "benign")
if MAX_PACKETS_BENIGN and _nt:
    print(f"  {_nt} benign captures were read only to their first {MAX_PACKETS_BENIGN:,} packets "
          f"(UNI_PACKET_CAP; set FULL = True to read them to the end).")

# A capture that yields no queries must say why here, not three cells later.
_dead = [s for s in PARSE if s["kind"] == "malicious" and s["dns"] == 0]
if _dead:
    print(f"\n  {len(_dead)} malicious capture(s) yielded no DNS queries:")
    for s in _dead:
        why = (s["error"] or
               (f"link type not readable ({s['linktypes']})" if s["unsupported_linktype"] else
                "no IP packets" if s["non_ip"] == s["packets"] else
                "no traffic on port 53" if s["port53"] == 0 else
                f"all {s['port53']:,} port-53 messages malformed"))
        print(f"    {s['file']}\n      packets {s['packets']:,}  link {s['linktypes'] or '-'}  "
              f"non-IP {s['non_ip']:,}  port-53 {s['port53']:,}  -> {why}\n"
              f"      first bytes: {s['first_bytes'][:80]}")
if not ben_queries:
    print("\n  WARNING: no benign DNS queries parsed -- attainment cells will be skipped or fail.")
if any(s["port53"] and s["malformed"] / s["port53"] > 0.05 for s in PARSE if s["kind"] == "malicious"):
    print("\n  A malformed share above 5% on port 53 usually means a tool tunnelling raw, "
          "non-DNS bytes over port 53 (iodine raw mode). Those bytes are NOT counted in the "
          "asset, so that tool's floors are measured on its DNS-borne payload only.")
bad_files = [s for s in PARSE if s["error"]]
trunc = [s for s in PARSE if s["truncated"]]
PKT_ERR = sum(s["packet_errors"] for s in PARSE); TCP_SPLIT = sum(s["tcp_split_read"] for s in PARSE)
print(f"\ncaptures that ended early (the container reader failed): {len(bad_files)}   "
      f"truncated at the packet cap (benign only): {len(trunc)}")
for s in bad_files[:10]:
    print(f"   {s['file']}\n      {s['error']}  -- after {s['packets']:,} packets, {s['seconds'] / 60:.1f} min of traffic")
print(f"packets that raised in the decoder and were skipped (the capture went on): {PKT_ERR:,}")
for s in [x for x in PARSE if x["packet_errors"]][:5]:
    print(f"   {s['file']}: {s['packet_errors']:,}   e.g. {s['packet_error_samples']}")
print(f"DNS-over-TCP messages whose 2-byte length arrived in a segment of its own, read from offset 0: {TCP_SPLIT:,}")
if USE_MAL and len(_dead) == len(USE_MAL):
    raise RuntimeError(
        f"None of the {len(USE_MAL)} malicious captures yielded a DNS query, so there is nothing "
        "to measure. The lines above give the reason for each file (link type, no port-53 "
        "traffic, malformed messages). Send that block back.")

In [ ]:
# ======================= EPISODES (with gap sensitivity) =======================
_require("mal_queries", "PARSE")
META = {d["rel"]: d for d in USE_MAL}
EP_LOG, episodes_by_gap = [], {}
for gap in GAP_GRID:
    eps = []
    for (tool, family), qs in sorted(mal_queries.items()):
        for e in build_episodes(list(qs), tool, gap_s=gap, min_queries=MIN_QUERIES,
                                min_payload=MIN_PAYLOAD_CHARS, family=family,
                                log=EP_LOG if gap == GAP_PRIMARY else None):
            m = META.get(e.meta["file"], {})
            e.meta.update(direction=m.get("direction", ""), rtype=m.get("rtype", ""))
            eps.append(e)
    episodes_by_gap[gap] = eps

print(f"{'gap (s)':>8s} {'episodes':>9s} {'families':>9s} {'median D (s)':>13s} {'median asset':>13s}")
for gap, eps in episodes_by_gap.items():
    if not eps:
        print(f"{gap:8.0f} {0:9d}"); continue
    ds = sorted(e.duration for e in eps); az = sorted(e.asset for e in eps)
    print(f"{gap:8.0f} {len(eps):9d} {len({e.family for e in eps}):9d} {ds[len(ds)//2]:13.1f} "
          f"{az[len(az)//2]:13,d}")

EPISODES = episodes_by_gap[GAP_PRIMARY]

# Per-capture decisions, printed BEFORE any stop so that a failure explains itself.
_dec = {r["file"]: r for r in EP_LOG if "tunnel_domain" in r}
_rej = defaultdict(list)
for r in EP_LOG:
    if r.get("rejected_run"):
        _rej[r["file"]].append(r)
_kept_n = Counter(e.meta["file"] for e in EPISODES)
_ps = {s["file"]: s for s in PARSE if s["kind"] == "malicious"}
WHY = {}
print(f"\nper-capture decisions (gap {GAP_PRIMARY:.0f} s; a run needs >= {MIN_QUERIES} queries "
      f"and >= {MIN_PAYLOAD_CHARS} payload chars):")
print(f"  {'capture':38s} {'DNS q':>8s} {'tunnel domain':24s} {'q in it':>8s} {'share':>6s} "
      f"{'bg dom':>6s} {'2-leg':>5s} {'eps':>4s}  verdict")
for d in USE_MAL:
    f = d["rel"]; s = _ps.get(f, {}); r = _dec.get(f, {}); rj = _rej.get(f, [])
    if not s:
        why = "not parsed"
    elif s["dns"] == 0:
        why = "no DNS queries parsed (see PARSE CAPTURES)"
    elif _kept_n[f]:
        why = "ok"
    elif r and r["n_queries_kept"] < MIN_QUERIES:
        why = (f"tunnel-domain group has only {r['n_queries_kept']} queries "
               f"(do the names share a registered domain?)")
    elif rj:
        big = max(rj, key=lambda x: (x["n_queries"], x["asset"]))
        why = (f"every run too small (largest: {big['n_queries']} queries, "
               f"{big['asset']:,} chars, {big['duration_s']:.1f} s)")
    else:
        why = "no run kept"
    WHY[f] = why
    print(f"  {os.path.basename(f)[:38]:38s} {s.get('dns', 0):8,d} "
          f"{r.get('tunnel_domain', '-')[:24]:24s} {r.get('n_queries_kept', 0):8,d} "
          f"{r.get('payload_share_kept', 0)*100:5.1f}% {r.get('n_background_domains', 0):6d} "
          f"{'yes' if r.get('duplicate_vantage_max_share', 0) > .5 else '':>5s} {_kept_n[f]:4d}  {why}")
print(f"  runs rejected as too small: {sum(len(v) for v in _rej.values())}")

if not EPISODES:
    _top = Counter(v.split(" (")[0].split(":")[0] for v in WHY.values()).most_common(3)
    raise RuntimeError(
        f"No episodes from the {len(USE_MAL)} selected malicious captures. The table above gives "
        "the reason for each file; most common: " + "; ".join(f"{k} ({n})" for k, n in _top) +
        ". Send that table and the PARSE CAPTURES table back.")
_miss = sorted({d["family"] for d in USE_MAL} - {e.family for e in EPISODES})
if _miss:
    print(f"\n  families with no episode (see verdicts above): {_miss}")

by_dir = Counter(e.meta["direction"] for e in EPISODES)
print(f"\nprimary frame: {len(EPISODES)} episodes across {len({e.family for e in EPISODES})} "
      f"families  (direction: {dict(by_dir)})")
print("manuscript's DAPT2021 base: 7 episodes, 1 dataset")

### The floor, at three pipeline positions on identical traffic

In [ ]:
# ============================== FLOORS ==============================
_require("EPISODES")

rows, notes = [], []
for ep in EPISODES:
    fl = {}
    for c in ORDER:
        f = measure_floor(ep, c, capture_end=ep.t1, **KW)
        if f:
            fl[c] = f; rows.append(f)
    notes += check_invariants(ep, fl, strict=False)
floors = pd.DataFrame(rows)

print(f"invariant notes: {len(notes)}")
for x in notes[:6]: print("  ", x)

print(f"\n{'consumer':10s} {'n':>4s} {'mean floor':>11s} {'95% CI (bootstrap)':>21s} "
      f"{'median':>8s} {'total loss':>11s} {'95% Wilson':>14s}")
print("-" * 86)
summary, FAMILY_FLOORS = [], {}
for c in ORDER:
    v = floors[floors.consumer == c].floor_measured.values
    if not len(v): continue
    lo, hi = bootstrap_ci(list(v), n_boot=N_BOOT, seed=SEED)
    k, n = int((v >= 0.999999).sum()), len(v)
    print(f"{c:10s} {n:4d} {v.mean()*100:10.3f}% [{lo*100:8.3f}, {hi*100:8.3f}]% "
          f"{np.median(v)*100:7.3f}% {k:5d}/{n:<5d} {wfmt(k, n):>14s}")
    # Family level. Episodes of one tool differ mainly by DNS record type, so n episodes is closer to the number of
    # families than to n independent observations: the family-weighted mean and its two-stage bootstrap (families,
    # then episodes within a family) are the numbers to quote. A family has "total loss" when every episode does.
    by_f = {f_: list(x_.floor_measured.values) for f_, x_ in floors[floors.consumer == c].groupby("family")}
    fm, flo, fhi = _fam(family_bootstrap, by_f, f"floor, {c}", n_boot=N_BOOT, seed=SEED)
    _, tlo, thi = _fam(family_t_interval, by_f, f"floor, {c}", clip=(0.0, 1.0))
    kf, nf = sum(1 for x_ in by_f.values() if min(x_) >= 0.999999), len(by_f)
    FAMILY_FLOORS[c] = {"mean": fm, "lo": flo, "hi": fhi, "t_lo": tlo, "t_hi": thi, "k": kf, "n": nf}
    summary.append({"consumer": c, "n": n, "mean_floor": v.mean(), "boot_lo": lo, "boot_hi": hi,
                    "median_floor": float(np.median(v)), "total_loss": k, "total_loss_frac": k / n,
                    "n_families": nf, "family_mean_floor": fm, "family_boot_lo": flo, "family_boot_hi": fhi,
                    "family_t_lo": tlo, "family_t_hi": thi, "families_total_loss": kf})
summary = pd.DataFrame(summary)
PQ_MEAN, FLOW_MEAN = (floors[floors.consumer == c].floor_measured.mean() for c in ("per-query", "flow"))
print(f"\nFAMILY LEVEL -- the numbers to quote ({floors.family.nunique()} families). Two intervals for the mean of the family "
      f"means: Student-t over the families (about 95% at this size), and the two-stage bootstrap (families, then episodes "
      f"inside a family), which at 8 families covers about 90%, not 95%, in simulation")
print(f"{'consumer':10s} {'family-weighted mean':>21s} {'t interval 95%':>22s} {'bootstrap (~90%)':>22s} {'families with total loss':>26s}")
for c, d in FAMILY_FLOORS.items():
    print(f"{c:10s} {d['mean']*100:20.3f}% [{d['t_lo']*100:7.3f}, {d['t_hi']*100:7.3f}]% [{d['lo']*100:7.3f}, {d['hi']*100:7.3f}]% "
          f"{d['k']:>10d}/{d['n']:<3d} Wilson {wfmt(d['k'], d['n'])}  exact {efmt(d['k'], d['n'])}")

print(f"\nby tool family (mean floor %, primary cadence):")
print(f"  {'family':18s} {'n':>3s} " + " ".join(f"{c:>10s}" for c in ORDER) + "   ports/episode")
for fam, g in floors.groupby("family"):
    vals = [g[g.consumer == c].floor_measured.mean() * 100 for c in ORDER]
    ports = g[g.consumer == "flow"].n_ports.median()
    print(f"  {fam:18s} {g.episode.nunique():3d} " + " ".join(f"{v:10.3f}" for v in vals) +
          f"   {ports:,.0f}")

A5_DELTA = floors[floors.consumer == "flow"].unit_delta_pp.abs()
print(f"\nA5 payload-unit sensitivity (flow floor, up to half of each typical query removed "
      f"as overhead):  median |delta| {A5_DELTA.median():.3f} pp, max {A5_DELTA.max():.3f} pp, "
      f"{int((A5_DELTA > 1).sum())} of {len(A5_DELTA)} episodes above 1 pp")

In [ ]:
# ============ measured floor vs the closed form min(1, T/D) ============
_require("floors")
s = floors[floors.consumer == "flow"]
CLOSED_EXC = s.excess_pp.values; CLOSED_BELOW = int((CLOSED_EXC < -1e-6).sum())
print(f"flow consumer, n = {len(s)}:  mean excess over min(1, T/D) {CLOSED_EXC.mean():+.2f} pp "
      f"(manuscript testbed: +0.7 pp, max +4.8);  below the closed form: {CLOSED_BELOW} of {len(s)}, "
      f"lowest {CLOSED_EXC.min():+.2f} pp")
if CLOSED_BELOW and CLOSED_EXC.min() < -0.5:
    print("  A back-loaded transfer legitimately sits below min(1, T/D): V(t) lags the "
          "diagonal. If this is material, the manuscript's 'sound and slightly optimistic' "
          "description of the closed form needs scoping to near-constant-rate transfers.")

In [ ]:
# ==================== CADENCE SWEEP: floor vs telemetry T ====================
_require("EPISODES")
sweep = []
for ep in EPISODES:
    for t in IDLE_GRID:
        f = measure_floor(ep, "flow", t_active=FLOW_ACTIVE_TIMEOUT, t_idle=t, capture_end=ep.t1)
        if f: f.update(param=t, axis="flow idle timeout"); sweep.append(f)
    # The ACTIVE timeout is what bounds a single long connection (floor = min(1, T_active / D)),
    # and vendor defaults span 60 s (Juniper) to 1800 s (Cisco). v3 swept idle and session only.
    for ta in ACTIVE_GRID:
        f = measure_floor(ep, "flow", t_active=ta, t_idle=IDLE_PRIMARY, capture_end=ep.t1)
        if f: f.update(param=ta, axis="flow active timeout"); sweep.append(f)
    for w in SESSION_GRID:
        f = measure_floor(ep, "session", window_s=w)
        if f: f.update(param=w, axis="session window"); sweep.append(f)
sweep = pd.DataFrame(sweep)
for axis in ("flow idle timeout", "flow active timeout", "session window"):
    a = sweep[sweep.axis == axis]
    print(f"\n{axis}\n  {'T (s)':>7s} {'mean floor':>11s} {'median':>9s} {'total loss':>11s}")
    for p, g in a.groupby("param"):
        v = g.floor_measured.values
        print(f"  {p:7.0f} {v.mean()*100:10.2f}% {np.median(v)*100:8.2f}% "
              f"{int((v >= .999999).sum()):5d}/{len(v):<5d}")

# Checkpoint. The floors are the science and need nothing from the models further down, so they are written
# NOW. A container that runs out of memory loses the kernel, not the files in the output folder: if a later
# cell is killed, these tables can still be sent back. The export cell rewrites the same files at the end.
def save_floor_tables():
    os.makedirs(OUT_DIR, exist_ok=True)
    floors.to_csv(f"{OUT_DIR}/dns_floors.csv", index=False)
    sweep.to_csv(f"{OUT_DIR}/dns_cadence_sweep.csv", index=False)
    summary.to_csv(f"{OUT_DIR}/dns_floor_summary.csv", index=False)
    pd.DataFrame(EP_LOG).to_csv(f"{OUT_DIR}/dns_episode_decisions.csv", index=False)
    pd.DataFrame([{k: v for k, v in s_.items() if k not in ("malformed_samples", "malformed_reason_counts")}
                  for s_ in PARSE]).to_csv(f"{OUT_DIR}/dns_parse_stats.csv", index=False)
    pd.DataFrame([{"file": s_["file"], "kind": s_["kind"], "source": s_.get("source", ""), "reason": r, "sample": x}
                  for s_ in PARSE for r, xs in s_["malformed_samples"].items() for x in xs],
                 columns=["file", "kind", "source", "reason", "sample"]).to_csv(
        f"{OUT_DIR}/dns_malformed_samples.csv", index=False)
    pd.DataFrame([{"gap_s": g, "n_episodes": len(e), "n_families": len({x.family for x in e})}
                  for g, e in episodes_by_gap.items()]).to_csv(f"{OUT_DIR}/dns_episode_sensitivity.csv", index=False)

save_floor_tables()
memory_mark("floors computed")
json.dump({"checkpoint": "written after the floors and the sweep; the attainment models had not run yet",
           "written": time.strftime("%Y-%m-%d %H:%M:%S"), "smoke": SMOKE, "full": FULL,
           "memory_guard": MEM_GUARD, "memory_in_use_gb": MEM_LOG, "n_episodes": len(EPISODES)},
          open(f"{OUT_DIR}/dns_checkpoint.json", "w"), indent=2, default=str)
print(f"\n  checkpoint: the floor tables, the parse statistics and the malformed-message samples are saved in "
      f"{OUT_DIR}. They stay there if a later cell is killed.")

### Attainment — do real detectors reach the floor?

The floor is what an *oracle* loses. Here real detectors are trained and measured at each
pipeline position under leave-one-family-out, with the a-priori Wilson operating point.
Invariant **I6** — no detector scores below its own floor — is asserted on every episode;
a violation means a bug or a false Proposition 1.

If held-out AUC is near 1, the attainment numbers measure the **floor itself**, with a
classifier good enough to stand in for the oracle — the narrowed reading the manuscript
adopted in §9. The AUC column is there so the two readings cannot be conflated.

In [ ]:
# ======================= CAUSAL FEATURES =======================
_require("EPISODES", "ben_queries", "PARSE", "USE_BEN")
QTYPES = [1, 5, 10, 15, 16, 28, 33, 255]

def _sub(name):
    """Name left of its registered domain: computable from one query, no look-ahead."""
    n = name.rstrip(".").lower(); rd = registered_domain(n)
    return n[:-len(rd) - 1] if n.endswith("." + rd) else ""

def featurise(qs):
    n = len(qs)
    subs = [_sub(q.qname) for q in qs]
    raw = [q.qname.rstrip(".").split(".")[0] for q in qs]
    sl = [len(s) for s in subs]
    ent = [shannon_entropy(s.replace(".", "")) for s in subs]
    labs = [s.count(".") + 1 if s else 0 for s in subs]
    mlab = [max((len(x) for x in s.split(".")), default=0) for s in subs]
    ts = [q.ts for q in qs]
    gaps = [b - a for a, b in zip(ts, ts[1:])] or [0.0]
    mean = lambda x: sum(x) / len(x)
    def std(x):
        m = mean(x)            # once per call. v3 recomputed it for EVERY element: quadratic in the
        return math.sqrt(sum((v - m) ** 2 for v in x) / len(x))   # unit size (131k queries ~ 4 min)
    s0, r0 = subs[0], raw[0]
    f = [n, ts[-1] - ts[0], mean(sl), std(sl), max(sl), mean(ent), std(ent), max(ent),
         mean(labs), max(mlab),
         sum(c.isdigit() for c in s0) / max(1, len(s0)),
         sum(c.isupper() for c in r0) / max(1, len(r0)),
         len(set(subs)) / n, mean(gaps), std(gaps), len({q.sport for q in qs}) / n]
    tc = Counter(q.qtype for q in qs)
    f += [tc.get(t, 0) / n for t in QTYPES] + [sum(v for k, v in tc.items() if k not in QTYPES) / n]
    return f

FEATNAMES = ["n_q", "span_s", "sub_len_mean", "sub_len_std", "sub_len_max", "ent_mean",
             "ent_std", "ent_max", "labels_mean", "max_label_len", "digit_ratio",
             "upper_ratio", "uniq_sub_ratio", "gap_mean", "gap_std", "port_ratio"] + \
            [f"qtype_{t}" for t in QTYPES] + ["qtype_other"]

# Guard against the v3 defect: std() recomputed its mean once per ELEMENT, so featurise() was
# quadratic in the size of a unit. A 134k-query session cost ~4 minutes per call and a "one
# minute" smoke run took 31. A linear featuriser costs microseconds per query.
_gq = max(EPISODES, key=lambda e: len(e.queries)).queries[:20_000]
_t = time.time(); featurise(_gq); _dt = time.time() - _t
print(f"featurise: {_dt / max(1, len(_gq)) * 1e6:.1f} us per query on a {len(_gq):,}-query unit")
if len(_gq) >= 10_000 and _dt / len(_gq) > 100e-6:
    raise RuntimeError(f"featurise() takes {_dt / len(_gq) * 1e6:.0f} us per query on a {len(_gq):,}-query unit: "
                       "it has become super-linear in unit size, and the run would take hours. "
                       "Look for a statistic that is recomputed inside a loop.")

# Benign units are built exactly like malicious episodes, then split BY CAPTURE FILE,
# stratified by source, so no capture contributes to both training and evaluation.
BEN_UNITS = benign_units(ben_queries, gap_s=GAP_PRIMARY)
SRC_OF = {d["rel"]: d["source"] for d in USE_BEN}
for u in BEN_UNITS: u.meta["source"] = SRC_OF.get(u.meta["file"], "benign")   # a capture's units share one meta dict
_release(); memory_mark("benign units built")
rng = random.Random(SEED)
TRAIN_FILES, TEST_FILES = set(), set()
for src in sorted(set(SRC_OF.values())):
    fs = sorted(f for f, s in SRC_OF.items() if s == src); rng.shuffle(fs)
    cut = max(1, len(fs) // 2) if len(fs) > 1 else 1
    TRAIN_FILES |= set(fs[:cut]); TEST_FILES |= set(fs[cut:]) if len(fs) > 1 else set(fs)
BEN_TRAIN = [u for u in BEN_UNITS if u.meta["file"] in TRAIN_FILES]
BEN_TEST = [u for u in BEN_UNITS if u.meta["file"] in TEST_FILES]
HOURS = {src: sum(s["seconds"] for s in PARSE if s.get("source") == src and s["file"] in TEST_FILES)
         / 3600.0 for src in set(SRC_OF.values())}
print(f"benign units: {len(BEN_UNITS):,} ({len(BEN_TRAIN):,} train / {len(BEN_TEST):,} held-out)  "
      f"held-out hours: " + ", ".join(f"{k} {v:.1f} h" for k, v in HOURS.items()))
IN_SAMPLE = {src for src in set(SRC_OF.values()) if sum(1 for v in SRC_OF.values() if v == src) < 2}
for src in sorted(IN_SAMPLE):
    print(f"  {src}: one capture only, so its train and held-out benign coincide -- its burdens are IN-SAMPLE "
          f"(expected in SMOKE; in a measurement run they would be optimistic) and are labelled so in the digest")

def unit_records(eps, consumer):
    out = []
    for ep in eps:
        for idxs, emit in CONSUMERS[consumer](ep, capture_end=ep.t1, **KW):
            out.append((ep, idxs, emit))
    return out

def matrix(recs, cap, seed=SEED):
    n_all = len(recs)
    if cap and n_all > cap:
        recs = random.Random(seed).sample(recs, cap)
    X = np.empty((len(recs), len(FEATNAMES)))               # filled row by row: no list-of-lists of 400k x 41 floats
    for j, (ep, idxs, _) in enumerate(recs):
        X[j] = featurise([ep.queries[i] for i in idxs])
    return X, recs, n_all

In [ ]:
# ================= LEAVE-ONE-FAMILY-OUT: TRAIN, OPERATING POINT, EBD =================
_require("EPISODES", "BEN_TRAIN", "BEN_TEST", "HOURS", "matrix", "unit_records", "featurise")
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

# Last line of defence. The parse cell sized the corpus to the container; this checks the actual state before the
# most allocation-heavy cell. At its peak training holds one record per benign query (about 200 bytes each) and
# the model needs a few hundred MB more. If that cannot fit, stop HERE with the reason and the fix, instead of
# letting the kernel be killed with nothing but "tried to allocate more memory". The floor tables are already
# saved (CADENCE SWEEP cell), so a stop here loses none of them.
_u_now, _lim_now = memory_mark("before training"), _mem_limit_bytes()
_need = 200 * len(ben_queries) + 0.5e9
if _lim_now and _u_now and _u_now + _need > 0.95 * _lim_now:
    raise RuntimeError(
        f"Stopped before training: {_u_now / 1e9:.1f} GB of {_lim_now / 1e9:.1f} GB are already in use and training needs "
        f"about {_need / 1e9:.1f} GB more, so the kernel would be killed. The floor tables are saved in {OUT_DIR} "
        f"(dns_floors.csv, dns_cadence_sweep.csv, dns_parse_stats.csv, dns_malformed_samples.csv): send those back. "
        "To train as well, lower N_UNIVERSITY_FILES in the CONFIGURATION cell (28 halves the university benign corpus), "
        "or set MEMORY_LIMIT_GB if this session really has more memory than was detected, and Run All again.")

try:                                              # a container can report every core of its host
    N_JOBS = max(1, min(8, len(os.sched_getaffinity(0))))
except Exception:
    N_JOBS = max(1, min(8, os.cpu_count() or 4))

def rf_model(pos, neg):
    return RandomForestClassifier(n_estimators=300, random_state=SEED, n_jobs=N_JOBS,
                                  class_weight="balanced")
try:
    from xgboost import XGBClassifier
    def xgb_model(pos, neg):
        return XGBClassifier(n_estimators=200, max_depth=6, learning_rate=0.1, subsample=0.9,
                             colsample_bytree=0.9, random_state=SEED, n_jobs=N_JOBS,
                             eval_metric="logloss", tree_method="hist",
                             scale_pos_weight=max(1.0, neg / max(1, pos)))
except Exception:
    xgb_model = None
FAMILIES = sorted({e.family for e in EPISODES})

def _limit_text(ls):
    """What set the thresholds of a group of folds, from limit_summary(): said in words so that 'the budget was binding'
    is only claimed where the budget was what stopped the threshold from going lower."""
    names = (("budget", "budget"), ("tie plateau", "tie plateau"), ("zero scores", "model's zero scores"), ("all fit", "all fit"),
             ("not certifiable", "NOT CERTIFIABLE (fewer than 381 training units: nothing alerts)"), ("over budget", "OVER BUDGET"))
    what = ", ".join(f"{lab} {ls[k]}" for k, lab in names if ls[k])
    used = (f"; budget used by the training units: median {ls['used_median'] * 100:.0f}% [{ls['used_min'] * 100:.0f}, "
            f"{ls['used_max'] * 100:.0f}]" if ls["used_median"] == ls["used_median"] else "")
    return f"threshold set by: {what} (of {ls['folds']} folds){used}"

def run_loto(new_model, model_name):
    attain, opp = [], []
    for consumer in ORDER:
        t_c = time.time()
        Xb_tr, _tr, nb_tr = matrix(unit_records(BEN_TRAIN, consumer), MAX_BENIGN_UNITS)
        te_recs = unit_records(BEN_TEST, consumer)
        Xb_te, te_s, nb_te = matrix(te_recs, MAX_BENIGN_UNITS)
        te_src = np.array([ep.meta["source"] for ep, _, _ in te_s])
        n_all_src = Counter(ep.meta["source"] for ep, _, _ in te_recs)
        del _tr, te_recs, te_s                                    # millions of tuples; only the matrices are needed
        _release(); memory_mark(f"{consumer}: benign features")
        mal_train = {}
        for ep in EPISODES:
            mal_train[ep.eid] = matrix(unit_records([ep], consumer), MAX_TRAIN_UNITS_PER_EPISODE)[0]
        for held in FAMILIES:
            tr = [e for e in EPISODES if e.family != held]
            te = [e for e in EPISODES if e.family == held]
            if not tr or not te or len(Xb_tr) == 0:
                continue
            Xm = np.vstack([mal_train[e.eid] for e in tr if len(mal_train[e.eid])])
            clf = new_model(len(Xm), len(Xb_tr))
            clf.fit(np.vstack([Xm, Xb_tr]), np.r_[np.ones(len(Xm)), np.zeros(len(Xb_tr))])
            # v4.2: the exact rule (no grid). The threshold is an observed training-benign score. `limit` says what set it
            # (the budget, a tie plateau, the model's own zeros, too few units to certify the budget) and `budget_used`
            # how much of the budget the training units use. v4.1 searched a grid whose lowest point (0.005) was already
            # inside the budget in 54 of 54 folds.
            thr, meta = pick_threshold(list(clf.predict_proba(Xb_tr)[:, 1]), budget=FPB_BUDGET)

            sb = clf.predict_proba(Xb_te)[:, 1] if len(Xb_te) else np.array([])
            row = {"consumer": consumer, "held_out_family": held, "model": model_name, "threshold": float(thr),
                   "train_burden": meta.get("train_burden", np.nan),
                   "train_upper": meta.get("train_upper", np.nan), "budget_bound": bool(meta.get("budget_bound") or False),
                   "limit": meta.get("limit", ""), "budget_used": meta.get("budget_used", np.nan),
                   "k_allowed": meta.get("k_allowed", -1), "n_train_benign": len(Xb_tr),
                   "n_heldout_benign_sampled": len(sb), "n_heldout_benign_total": nb_te}
            for src in sorted(set(te_src)) + ["pooled"]:
                mask = np.ones(len(sb), bool) if src == "pooled" else (te_src == src)
                k, n = int((sb[mask] >= thr).sum()), int(mask.sum())
                lo, hi = wilson(k, n)
                n_tot = sum(n_all_src.values()) if src == "pooled" else n_all_src[src]
                hrs = sum(HOURS.values()) if src == "pooled" else HOURS.get(src, 0)
                row.update({f"fpb_{src}": k / n if n else np.nan, f"fpb_{src}_lo": lo,
                            f"fpb_{src}_hi": hi, f"over_budget_{src}": bool(n and lo > FPB_BUDGET), f"n_heldout_{src}": n,
                            f"alerts_per_hour_{src}": (k / n) * n_tot / hrs if n and hrs else np.nan})
            row["within_budget"] = bool(row.get("fpb_pooled", 1) <= FPB_BUDGET)

            recs = unit_records(te, consumer)
            Xm_te = np.array([featurise([ep.queries[i] for i in idxs]) for ep, idxs, _ in recs])
            sm = clf.predict_proba(Xm_te)[:, 1] if len(Xm_te) else np.array([])
            row["auc"] = (roc_auc_score(np.r_[np.ones(len(sm)), np.zeros(len(sb))], np.r_[sm, sb])
                          if len(sm) and len(sb) else np.nan)
            opp.append(row)
            lut = {(ep.eid, idxs[0]): float(s) for (ep, idxs, _), s in zip(recs, sm)}
            for ep in te:
                r = measure_ebd(ep, consumer, lambda e, i, t: lut.get((e.eid, i[0]), 0.0), thr,
                                capture_end=ep.t1, **KW)
                r.update(held_out_family=held, threshold=thr, auc=row["auc"],
                         fpb_pooled=row.get("fpb_pooled", np.nan))
                check_invariants(ep, {consumer: r}, strict=False)          # I6 raises
                attain.append(r)
        del Xb_tr, Xb_te, mal_train; _release(); memory_mark(f"{consumer}: folds done")
        _o = [r for r in opp if r["consumer"] == consumer]
        print(f"  {consumer:10s} done in {time.time()-t_c:5.0f}s" + (
            f"   threshold {min(r['threshold'] for r in _o):.3g} .. {max(r['threshold'] for r in _o):.3g}; "
            f"{_limit_text(limit_summary(_o))}" if _o else ""))
    return pd.DataFrame(attain), pd.DataFrame(opp)

if xgb_model is not None:
    MODEL = "XGBoost"
    print(f"model: {MODEL}   families: {FAMILIES}")
    try:
        attain, opp = run_loto(xgb_model, MODEL)
    except InvariantViolation:
        raise
    except Exception as e:
        MODEL = f"RandomForest (XGBoost failed: {type(e).__name__})"
        print(f"\n  XGBoost failed ({type(e).__name__}: {e}).\n  Re-running EVERY fold with "
              f"RandomForest, so all folds share one model.")
        attain, opp = run_loto(rf_model, MODEL)
else:
    MODEL = "RandomForest (xgboost unavailable)"
    print(f"model: {MODEL}   families: {FAMILIES}")
    attain, opp = run_loto(rf_model, MODEL)
print(f"\nI6 held on all {len(attain)} episode-detector pairs: no detector scored below its floor.")

In [ ]:
# ============ ibHH-STYLE STREAMING ACCUMULATOR (outside the family) ============
_require("EPISODES", "BEN_TRAIN", "BEN_TEST", "HOURS")
# Isolated: a failure here is reported and the rest of the run stands.
ibhh = pd.DataFrame(); IBHH_META = {}
if RUN_IBHH:
    try:
        t_c = time.time()
        # v3 calibrated on all benign streams POOLED. Pooling lets an easy source dilute the budget:
        # in the 4 Oct smoke run the pooled burden was 0.89% but the university alone was 1.43%
        # [1.29, 1.59] -- over budget on the one realistic source, and a full run (68 top-1M files
        # against the university's streams) would have diluted it further. v4 takes the LARGEST of
        # the per-source thresholds, so the budget holds on every benign source. The pooled-rule
        # threshold is kept, and scored, so the size of the effect is on the record.
        MIN_STREAMS = 400   # below this, 0 events in n streams already has a Wilson upper bound > 1%
        train_by_src = defaultdict(list)
        for u in BEN_TRAIN:
            train_by_src[u.meta["source"]].append(u.queries)
        per_src = {s: ibhh_threshold(q, IBHH_WINDOW_S, FPB_BUDGET)
                   for s, q in train_by_src.items() if len(q) >= MIN_STREAMS}
        thr_pooled, meta_pooled = ibhh_threshold([u.queries for u in BEN_TRAIN], IBHH_WINDOW_S, FPB_BUDGET)
        if per_src:
            binding = max(per_src, key=lambda s: per_src[s][0])
            thr_i, meta_i = per_src[binding]
            meta_i = {**meta_i, "calibrated_on": f"max over sources; binding source: {binding}"}
        else:
            binding, thr_i = "pooled", thr_pooled
            meta_i = {**meta_pooled, "calibrated_on": "pooled (no source has enough streams alone)"}
        fired = Counter(); fired_pr = Counter(); total = Counter()
        for u in BEN_TEST:
            s_ = u.meta["source"]; total[s_] += 1
            mx = max((v for _, v in unique_info_trace(u.queries, IBHH_WINDOW_S)), default=0)
            if mx >= thr_i: fired[s_] += 1
            if mx >= thr_pooled: fired_pr[s_] += 1
        IBHH_META = {"threshold_chars": thr_i, "window_s": IBHH_WINDOW_S, **meta_i,
                     "threshold_pooled_rule": thr_pooled, "threshold_by_source": {s: t for s, (t, _) in per_src.items()},
                     "n_benign_streams_pooled": meta_pooled.get("n_benign_streams", 0)}
        for src in sorted(total):
            lo, hi = wilson(fired[src], total[src]); lo_p, hi_p = wilson(fired_pr[src], total[src])
            IBHH_META.update({f"fpb_{src}": fired[src] / total[src], f"fpb_{src}_lo": lo,
                              f"fpb_{src}_hi": hi, f"over_budget_{src}": bool(lo > FPB_BUDGET),
                              f"fpb_{src}_pooled_rule": fired_pr[src] / total[src],
                              f"fpb_{src}_pooled_rule_lo": lo_p, f"fpb_{src}_pooled_rule_hi": hi_p,
                              f"alerts_per_hour_{src}": fired[src] / HOURS[src] if HOURS.get(src) else np.nan})
        k, n = sum(fired.values()), sum(total.values())
        IBHH_META.update(fpb_pooled=k / n if n else np.nan,
                         alerts_per_hour_pooled=k / sum(HOURS.values()) if sum(HOURS.values()) else np.nan)
        rows = []
        for ep in EPISODES:
            t = ibhh_alert_time(ep, thr_i, IBHH_WINDOW_S)
            ebd = 1.0 if t is None else fraction_lost_by(ep, t)
            pqf = measure_floor(ep, "per-query")["floor_measured"]
            fwf = measure_floor(ep, "flow", capture_end=ep.t1, **KW)["floor_measured"]
            if ebd + 1e-9 < pqf:
                raise InvariantViolation(f"I6 ibHH-style EBD below its per-query floor on {ep.eid}")
            rows.append({"episode": ep.eid, "family": ep.family, "direction": ep.meta["direction"],
                         "alerted": t is not None, "ebd": ebd, "floor_per_query": pqf,
                         "floor_flow": fwf, "statistical_delay_pp": (ebd - pqf) * 100,
                         "below_flow_floor": ebd < fwf - 1e-12, "flow_floor_total": fwf >= 0.999999})
        ibhh = pd.DataFrame(rows)
        print(f"threshold {thr_i:,.0f} unique subdomain chars per {IBHH_WINDOW_S:.0f} s  "
              f"({meta_i['calibrated_on']}; per-source {IBHH_META['threshold_by_source']}; "
              f"the v3 pooled rule would give {thr_pooled:,.0f})")
        print("held-out benign, this rule:   " + ", ".join(f"{s} {IBHH_META[f'fpb_{s}']*100:.3f}% "
              f"[{IBHH_META[f'fpb_{s}_lo']*100:.3f}, {IBHH_META[f'fpb_{s}_hi']*100:.3f}]"
              + ("  <-- OVER BUDGET" if IBHH_META[f'over_budget_{s}'] else "") for s in sorted(total)))
        print("held-out benign, pooled rule: " + ", ".join(f"{s} {IBHH_META[f'fpb_{s}_pooled_rule']*100:.3f}% "
              f"[{IBHH_META[f'fpb_{s}_pooled_rule_lo']*100:.3f}, {IBHH_META[f'fpb_{s}_pooled_rule_hi']*100:.3f}]"
              for s in sorted(total)))
        if any(IBHH_META[f'over_budget_{s}'] for s in total):
            print("  A source is over budget with the per-source rule too: its held-out streams differ "
                  "from its training streams. Reported, never retuned on held-out data.")
        k_b = int(ibhh.below_flow_floor.sum()); n_e = len(ibhh)
        _inf = ibhh[~ibhh.flow_floor_total]; k_i, n_i = int(_inf.below_flow_floor.sum()), len(_inf)
        print(f"EBD mean {ibhh.ebd.mean()*100:.3f}%   statistical delay over the per-query floor "
              f"mean {ibhh.statistical_delay_pp.mean():.3f} pp")
        print(f"below the FLOW floor (escapes the record-consuming bound): {k_b}/{n_e} "
              f"{wfmt(k_b, n_e)}   never alerted: {int((~ibhh.alerted).sum())}   "
              f"({time.time()-t_c:.0f}s)")
        print(f"  but {n_e - n_i} of those {n_e} episodes have a flow floor of 100% (the whole transfer is over before any "
              f"flow record exists), where any early alert is 'below' it. The informative comparison is the {n_i} episodes "
              f"whose flow floor is under 100%: {k_i}/{n_i} {wfmt(k_i, n_i)}")
    except Exception as e:
        print(f"ibHH-style comparator FAILED ({type(e).__name__}: {e}); the rest of the run stands.")
        if isinstance(e, InvariantViolation): raise

In [ ]:
# ============================ ATTAINMENT RESULTS ============================
_require("attain", "opp", "ibhh", "IBHH_META", "IN_SAMPLE")
def _lab(src): return src + (" (in-sample)" if src in IN_SAMPLE else "")

print(f"{'consumer':10s} {'n':>4s} {'mean floor':>11s} {'mean EBD':>10s} {'gap pp':>8s} "
      f"{'max gap':>8s} {'at floor':>10s} {'95% Wilson':>14s} {'<=tol':>7s} {'AUC':>6s} {'never fired':>12s}")
print("-" * 112)
print(f"  (at floor = |EBD - floor| < 1e-9, the strict rule.  <=tol = gap <= {AT_FLOOR_TOL_PP} pp, the "
      f"order of the A5 payload-unit uncertainty: a per-query detector that fires on the 2nd or 3rd "
      f"query is 'not at floor' by the strict rule and indistinguishable from it by the second.)")
att_summary, FAMILY_ATT = [], {}
for c in ORDER:
    s = attain[attain.consumer == c] if len(attain) else attain
    if not len(s): continue
    k, n = int((s.at_floor).sum()), len(s)
    kt = int((s.gap_pp <= AT_FLOOR_TOL_PP).sum())
    print(f"{c:10s} {n:4d} {s.floor_measured.mean()*100:10.3f}% {s.ebd.mean()*100:9.3f}% "
          f"{s.gap_pp.mean():8.3f} {s.gap_pp.max():8.3f} {k:5d}/{n:<4d} {wfmt(k, n):>14s} "
          f"{kt:3d}/{n:<3d} {s.auc.mean():6.3f} {int((~s.alerted).sum()):12d}")
    # Family level, as for the floors: families are resampled, then episodes inside a family. The same seed and the
    # same family sizes give the same resamples, so the EBD, floor and gap intervals describe one set of draws.
    by_e = {f_: list(x_.ebd.values) for f_, x_ in s.groupby("family")}
    by_g = {f_: list(x_.gap_pp.values) for f_, x_ in s.groupby("family")}
    fe = _fam(family_bootstrap, by_e, f"EBD, {c}", n_boot=N_BOOT, seed=SEED)
    fg = _fam(family_bootstrap, by_g, f"gap, {c}", n_boot=N_BOOT, seed=SEED)
    te = _fam(family_t_interval, by_e, f"EBD, {c}", clip=(0.0, 1.0))
    tg = _fam(family_t_interval, by_g, f"gap, {c}", clip=(0.0, None))
    fam_ok = sum(1 for _, x_ in s.groupby("family") if (x_.gap_pp <= AT_FLOOR_TOL_PP).all())     # every episode of the family
    FAMILY_ATT[c] = {"ebd": fe, "gap": fg, "ebd_t": te, "gap_t": tg, "within_tol": fam_ok, "n": s.family.nunique()}
    att_summary.append({"consumer": c, "n": n, "mean_floor": s.floor_measured.mean(),
                        "mean_ebd": s.ebd.mean(), "mean_gap_pp": s.gap_pp.mean(),
                        "max_gap_pp": s.gap_pp.max(), "at_floor": k,
                        "at_floor_lo": wilson(k, n)[0], "at_floor_hi": wilson(k, n)[1],
                        "within_tol": kt, "within_tol_lo": wilson(kt, n)[0], "within_tol_hi": wilson(kt, n)[1],
                        "mean_auc": s.auc.mean(), "never_alerted": int((~s.alerted).sum()),
                        "n_families": s.family.nunique(), "family_mean_ebd": fe[0], "family_ebd_lo": fe[1],
                        "family_ebd_hi": fe[2], "family_mean_gap_pp": fg[0], "family_gap_lo": fg[1],
                        "family_gap_hi": fg[2], "family_ebd_t_lo": te[1], "family_ebd_t_hi": te[2],
                        "family_gap_t_lo": tg[1], "family_gap_t_hi": tg[2], "families_within_tol": fam_ok})
if len(ibhh):
    print(f"{'ibHH-style':10s} {len(ibhh):4d} {ibhh.floor_per_query.mean()*100:10.3f}% "
          f"{ibhh.ebd.mean()*100:9.3f}% {ibhh.statistical_delay_pp.mean():8.3f}   (per-query "
          f"floor; gap = statistical delay)")
att_summary = pd.DataFrame(att_summary)

if len(att_summary):
    print(f"\nFAMILY LEVEL -- the numbers to quote: the mean of the family means, with a Student-t interval over the families "
          f"(about 95%) and the two-stage bootstrap (families, then episodes within a family; about 90% at 8 families)")
    print(f"  {'consumer':10s} {'EBD %':>10s} {'t 95%':>22s} {'bootstrap':>22s}   {'gap pp':>9s} {'t 95%':>20s} {'bootstrap':>20s}")
    for c, d in FAMILY_ATT.items():
        print(f"  {c:10s} {d['ebd'][0]*100:10.3f} [{d['ebd_t'][1]*100:8.3f}, {d['ebd_t'][2]*100:8.3f}] "
              f"[{d['ebd'][1]*100:8.3f}, {d['ebd'][2]*100:8.3f}]   {d['gap'][0]:9.3f} [{d['gap_t'][1]:7.3f}, {d['gap_t'][2]:7.3f}] "
              f"[{d['gap'][1]:7.3f}, {d['gap'][2]:7.3f}]")
    print(f"  families with EVERY episode within {AT_FLOOR_TOL_PP} pp of the floor: " + ", ".join(
          f"{c} {d['within_tol']}/{d['n']} Wilson {wfmt(d['within_tol'], d['n'])} exact {efmt(d['within_tol'], d['n'])}"
          for c, d in FAMILY_ATT.items()))
    print(f"\nby held-out family: EBD % (gap pp over the floor)")
    _fams = sorted(attain.family.unique())
    print(f"  {'family':18s} " + " ".join(f"{c:>20s}" for c in ORDER))
    for f_ in _fams:
        cells_ = []
        for c in ORDER:
            g_ = attain[(attain.consumer == c) & (attain.family == f_)]
            cells_.append(f"{g_.ebd.mean()*100:9.3f} ({g_.gap_pp.mean():7.3f})" if len(g_) else f"{'-':>18s}")
        print(f"  {f_:18s} " + " ".join(f"{x:>20s}" for x in cells_))

srcs = sorted({c[4:] for c in opp.columns if c.startswith("fpb_") and not c.endswith(("_lo", "_hi"))}) if len(opp) else []
_W = max([24] + [len(_lab(s)) for s in srcs])
print(f"\nheld-out benign burden (mean over folds) and alerts per monitored hour (n/a: no held-out units from that source):")
print(f"  {'consumer':10s} " + " ".join(f"{_lab(s):>{_W}s}" for s in srcs))
for c in (ORDER if len(opp) else []):
    g = opp[opp.consumer == c]
    if not len(g): continue
    print(f"  {c:10s} " + " ".join(
        f"{_pc(g[f'fpb_{s}'].mean()) + '  ' + _ph(g[f'alerts_per_hour_{s}'].mean()):>{_W}s}" for s in srcs))
if IBHH_META:
    print(f"  {'ibHH-style':10s} " + " ".join(
        f"{_pc(IBHH_META.get(f'fpb_{s}', float('nan'))) + '  ' + _ph(IBHH_META.get(f'alerts_per_hour_{s}', float('nan'))):>{_W}s}"
        for s in srcs))
if IN_SAMPLE:
    print(f"  (in-sample: {sorted(IN_SAMPLE)} has one capture, so its 'held-out' benign is also what the model and the "
          f"threshold were fitted on)")

if len(opp):
    print(f"\noperating points (exact rule: the lowest observed training-benign score that keeps the upper Wilson bound "
          f"inside {FPB_BUDGET*100:.0f}%)")
    print(f"  {'consumer':10s} {'folds':>6s} {'threshold range':>26s} {'train burden (mean)':>20s}   what set the threshold")
    for c in ORDER:
        g = opp[opp.consumer == c]
        if not len(g): continue
        print(f"  {c:10s} {len(g):6d} {g.threshold.min():>12.3g} .. {g.threshold.max():<10.3g} {g.train_burden.mean()*100:18.3f}%   "
              f"{_limit_text(limit_summary(g.to_dict('records')))}")
over = opp[~opp.within_budget] if len(opp) else opp
print(f"\nfolds over the {FPB_BUDGET*100:.0f}% budget on held-out benign, pooled point estimate: {len(over)} of {len(opp)} "
      f"(reported, never retuned on held-out data)")
OVER_SRC = {}
for s in srcs:
    if s != "pooled" and f"over_budget_{s}" in opp.columns:
        OVER_SRC[s] = {}
        for c in ORDER:
            g_ = opp[opp.consumer == c]
            if not len(g_): continue
            has = (g_[f"n_heldout_{s}"] > 0) if f"n_heldout_{s}" in g_.columns else pd.Series(True, index=g_.index)
            OVER_SRC[s][c] = (int(g_[has][f"over_budget_{s}"].sum()), int(has.sum()))      # folds with no held-out unit do not count
        print(f"  {_lab(s)}: folds whose Wilson LOWER bound exceeds the budget -- " +
              ", ".join(f"{c} {k}/{n}" if n else f"{c} n/a (no held-out units)" for c, (k, n) in OVER_SRC[s].items()))
print("A percentage burden is per unit, and a per-query detector has far more units per hour "
      "than a session detector -- compare granularities on alerts/hour, not percentage.")

In [ ]:
# ============================ EXPORT + DIGEST ============================
_require("floors", "summary", "sweep", "save_floor_tables", "CLOSED_EXC", "A5_DELTA", "EP_LOG", "PARSE", "MEM_GUARD", "attain",
         "opp", "att_summary", "srcs", "over", "ibhh", "IBHH_META", "episodes_by_gap", "MODEL", "FAMILY_FLOORS",
         "FAMILY_ATT", "OVER_SRC", "IN_SAMPLE")
save_floor_tables()                              # defined in the sweep cell; also written there as a checkpoint
if len(attain):
    attain.to_csv(f"{OUT_DIR}/dns_attainment.csv", index=False)
    att_summary.to_csv(f"{OUT_DIR}/dns_attainment_summary.csv", index=False)
    opp.to_csv(f"{OUT_DIR}/dns_operating_points.csv", index=False)
if len(ibhh):
    ibhh.to_csv(f"{OUT_DIR}/dns_ibhh.csv", index=False)

prov = {"version": VERSION, "dataset": "daumel/dns-tunneling-dataset", "started": RUN_STARTED,
        "finished": time.strftime("%Y-%m-%d %H:%M:%S"), "smoke": SMOKE, "full": FULL,
        "seed": SEED, "model": MODEL, "directions_included": list(DIRECTIONS_INCLUDED),
        "n_down_captures_excluded": len(EXCLUDED_DOWN),
        "n_malicious_captures": len(USE_MAL), "n_benign_captures": len(USE_BEN),
        "n_university_captures": sum(d in uni for d in USE_BEN), "n_university_available": len(uni),
        "n_packets": int(sum(s["packets"] for s in PARSE)),
        "n_parse_errors": len(bad_files), "n_captures_ended_early": len(bad_files), "n_packet_errors": PKT_ERR,
        "n_tcp_split_read": TCP_SPLIT, "n_benign_truncated": len(trunc),
        "in_sample_sources": sorted(IN_SAMPLE),
        "uni_packet_cap": MAX_PACKETS_BENIGN, "active_grid_s": ACTIVE_GRID, "at_floor_tol_pp": AT_FLOOR_TOL_PP,
        "cell_seconds": CELL_SECONDS, "total_seconds": round(sum(t for _, t in CELL_SECONDS), 1),
        "cell_peak_mb": CELL_PEAK_MB, "peak_memory_mb": max([m for m in CELL_PEAK_MB if m] or [None], default=None),
        "memory_limit": {"gb": MEM_GUARD["limit_gb"], "source": MEM_GUARD["limit_source"], "raw": MEM_GUARD["limit_raw"]},
        "memory_guard": MEM_GUARD, "memory_in_use_gb": MEM_LOG, "mem_frac_parse": MEM_FRAC_PARSE,
        "n_jobs": N_JOBS if "N_JOBS" in globals() else None,
        "gap_primary_s": GAP_PRIMARY, "idle_primary_s": IDLE_PRIMARY,
        "session_primary_s": SESSION_PRIMARY, "flow_active_timeout_s": FLOW_ACTIVE_TIMEOUT,
        "fpb_budget": FPB_BUDGET, "max_benign_units": MAX_BENIGN_UNITS,
        "max_train_units_per_episode": MAX_TRAIN_UNITS_PER_EPISODE,
        "n_episodes": len(EPISODES), "n_families": len({e.family for e in EPISODES}),
        "family_floors": FAMILY_FLOORS, "family_attainment": FAMILY_ATT,
        "ibhh": IBHH_META, "invariant_notes": notes[:200]}
def _jsonable(o):
    """Strict JSON: NaN and +-Infinity (a source with no held-out units, a threshold that cannot be certified) become strings."""
    if isinstance(o, float):
        return o if math.isfinite(o) else str(o)
    if isinstance(o, dict):
        return {str(k): _jsonable(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)):
        return [_jsonable(v) for v in o]
    return o
json.dump(_jsonable(prov), open(f"{OUT_DIR}/dns_provenance.json", "w"), indent=2, default=str, allow_nan=False)
try:
    os.remove(f"{OUT_DIR}/dns_checkpoint.json")        # a complete run supersedes the checkpoint
except OSError:
    pass

def _srcl(s_): return s_ + ("(in-sample)" if s_ in IN_SAMPLE else "")
def _ci(d, scale=100.0, nd=3): return f"{d[0]*scale:.{nd}f}% [{d[1]*scale:.{nd}f}, {d[2]*scale:.{nd}f}]"

L = []
L.append(f"EBD-DNS RESULTS DIGEST  v{VERSION}  {prov['finished']}  SMOKE={SMOKE} FULL={FULL} model={MODEL}")
L.append(f"corpus  malicious {len(USE_MAL)} captures ({len(EXCLUDED_DOWN)} _down excluded); benign "
         f"{len(USE_BEN)} (university {prov['n_university_captures']}/{len(uni)}); "
         f"{prov['n_packets']:,} packets")
L.append(f"parser  captures ended early {len(bad_files)}; packets skipped after an error {PKT_ERR:,}; "
         f"DNS-over-TCP length-prefix-alone reads {TCP_SPLIT:,}")
for s_ in bad_files[:6]:
    L.append(f"  ENDED EARLY  {s_['file']}: {s_['error']} after {s_['packets']:,} packets "
             f"({s_['seconds'] / 60:.1f} min of traffic)")
_bm = []
for src_ in sorted({s_["source"] for s_ in PARSE if s_["kind"] == "benign"}):
    ss_ = [s_ for s_ in PARSE if s_["kind"] == "benign" and s_["source"] == src_]
    p_ = sum(s_["port53"] for s_ in ss_); b_ = sum(s_["malformed"] for s_ in ss_)
    q_ = sum(s_["dns"] for s_ in ss_); bq_ = sum(s_["malformed_c2s"] for s_ in ss_)
    why_ = Counter()
    for s_ in ss_: why_.update(s_["malformed_reason_counts"])
    _bm.append(f"{src_} {b_ / max(1, p_) * 100:.1f}% of port-53 messages ({bq_ / max(1, bq_ + q_) * 100:.1f}% of those sent to port 53)"
               + (f", most common: {why_.most_common(1)[0][0]}" if why_ else ""))
L.append("benign messages excluded as malformed: " + ("; ".join(_bm) if _bm else "none (no benign captures)") + ";  " +
         (f"each benign capture read to its first {MAX_PACKETS_BENIGN:,} packets" if MAX_PACKETS_BENIGN
          else "each benign capture read to its end"))
L.append(f"episodes  {len(EPISODES)} across {prov['n_families']} families at gap {GAP_PRIMARY:.0f}s; "
         "by gap " + ", ".join(f"{g:.0f}s:{len(e)}" for g, e in episodes_by_gap.items()))
L.append("floor, episode level (treats episodes as independent; see the family level below)  "
         "consumer  n  mean [boot 95%]  median  total-loss k/n [Wilson]")
for _, r in summary.iterrows():
    L.append(f"  {r.consumer:10s} {r.n:3d}  {r.mean_floor*100:.3f}% [{r.boot_lo*100:.3f}, "
             f"{r.boot_hi*100:.3f}]  {r.median_floor*100:.3f}%  {r.total_loss}/{r.n} {wfmt(r.total_loss, r.n)}")
L.append(f"floor, FAMILY level (quote these: {prov['n_families']} families)  consumer  family-weighted mean  "
         "[Student-t 95% over the families]  [two-stage bootstrap, about 90% at 8 families]  families with total loss k/m "
         "[Wilson] [exact]")
for c_, d_ in FAMILY_FLOORS.items():
    L.append(f"  {c_:10s} {d_['mean']*100:.3f}%  t [{d_['t_lo']*100:.3f}, {d_['t_hi']*100:.3f}]  "
             f"boot [{d_['lo']*100:.3f}, {d_['hi']*100:.3f}]  {d_['k']}/{d_['n']} {wfmt(d_['k'], d_['n'])} {efmt(d_['k'], d_['n'])}")
L.append(f"closed form: mean excess {CLOSED_EXC.mean():+.2f}pp, below {CLOSED_BELOW}/{len(CLOSED_EXC)} "
         f"(lowest {CLOSED_EXC.min():+.2f}pp) | A5 unit: median |d| {A5_DELTA.median():.3f}pp, max {A5_DELTA.max():.3f}pp")
_ac = sweep[sweep.axis == "flow active timeout"]
if len(_ac):
    L.append("flow floor by ACTIVE timeout (mean %, total loss k/n)  " + "  ".join(
        f"{p_:.0f}s:{g_.floor_measured.mean() * 100:.2f}% {int((g_.floor_measured >= .999999).sum())}/{len(g_)}"
        for p_, g_ in _ac.groupby("param")))
fam_line = []
for fam, g in floors.groupby("family"):
    fam_line.append(f"{fam}(n={g.episode.nunique()}):" + "/".join(f"{g[g.consumer == c].floor_measured.mean()*100:.2f}" for c in ORDER))
L.append("floor by family (pq/flow/session %)  " + "  ".join(fam_line))
if len(att_summary):
    L.append("attainment (leave-one-family-out, exact Wilson-upper operating point)")
    for _, r in att_summary.iterrows():
        g = opp[opp.consumer == r.consumer]
        burden = "  ".join(f"{_srcl(s)}:{_pc(g[f'fpb_{s}'].mean())}/{_ph(g[f'alerts_per_hour_{s}'].mean(), 'ph')}"
                           for s in srcs if s != "pooled")
        fa = FAMILY_ATT[r.consumer]
        L.append(f"  {r.consumer:10s} within {AT_FLOOR_TOL_PP}pp of the floor {r.within_tol}/{r.n} [{r.within_tol_lo*100:.1f}, "
                 f"{r.within_tol_hi*100:.1f}]  never alerted {r.never_alerted}  gap max {r.max_gap_pp:.3f}pp  "
                 f"AUC {r.mean_auc:.3f}  (strict at-floor {r.at_floor}/{r.n})")
        L.append(f"    family level: EBD {fa['ebd'][0]*100:.3f}%  t [{fa['ebd_t'][1]*100:.3f}, {fa['ebd_t'][2]*100:.3f}]  "
                 f"boot [{fa['ebd'][1]*100:.3f}, {fa['ebd'][2]*100:.3f}]   gap {fa['gap'][0]:.3f}pp  t [{fa['gap_t'][1]:.3f}, "
                 f"{fa['gap_t'][2]:.3f}]  boot [{fa['gap'][1]:.3f}, {fa['gap'][2]:.3f}]   families with every episode within "
                 f"{AT_FLOOR_TOL_PP}pp {fa['within_tol']}/{fa['n']} {wfmt(fa['within_tol'], fa['n'])} {efmt(fa['within_tol'], fa['n'])}")
        L.append(f"    operating point: threshold {g.threshold.min():.3g} .. {g.threshold.max():.3g}; "
                 f"{_limit_text(limit_summary(g.to_dict('records')))}; training burden {g.train_burden.mean()*100:.3f}% "
                 f"(upper {g.train_upper.mean()*100:.3f}%)")
        L.append(f"    held-out burden / alerts per hour: {burden}")
    L.append("  EBD by held-out family (pq/flow/session %)  " + "  ".join(
        f"{f_}:" + "/".join(f"{attain[(attain.consumer == c) & (attain.family == f_)].ebd.mean()*100:.3f}" for c in ORDER)
        for f_ in sorted(attain.family.unique())))
    L.append(f"  folds over budget on held-out benign (pooled point estimate): {len(over)}/{len(opp)}")
    for s_, d_ in OVER_SRC.items():
        L.append(f"  folds with the Wilson LOWER bound over budget, {_srcl(s_)} (folds with held-out units only): " +
                 ", ".join(f"{c_} {k_}/{n_}" if n_ else f"{c_} n/a" for c_, (k_, n_) in d_.items()))
if len(ibhh):
    kb = int(ibhh.below_flow_floor.sum()); _inf = ibhh[~ibhh.flow_floor_total]
    ki, ni = int(_inf.below_flow_floor.sum()), len(_inf)
    L.append(f"ibHH-style W={IBHH_WINDOW_S:.0f}s thr={IBHH_META['threshold_chars']:.0f} "
             f"({IBHH_META['calibrated_on']}; the v3 pooled rule would give {IBHH_META['threshold_pooled_rule']:.0f}): "
             f"EBD mean {ibhh.ebd.mean()*100:.3f}%  stat-delay {ibhh.statistical_delay_pp.mean():.3f}pp  never alerted {int((~ibhh.alerted).sum())}")
    L.append(f"  below the flow floor {kb}/{len(ibhh)}, of which {kb - ki} have a flow floor of 100% (trivially below); "
             f"INFORMATIVE comparison (flow floor under 100%): {ki}/{ni} {wfmt(ki, ni)}")
    L.append("  ibHH held-out FPB, this rule: " + "  ".join(
        f"{_srcl(s)}:{_pc(IBHH_META.get(f'fpb_{s}', float('nan')))}" + ("(OVER)" if IBHH_META.get(f'over_budget_{s}') else "")
        for s in srcs if s != "pooled") + "   pooled rule: " + "  ".join(
        f"{_srcl(s)}:{_pc(IBHH_META.get(f'fpb_{s}_pooled_rule', float('nan')))}" for s in srcs if s != "pooled")
        + "   (a 60 s stream is not the learned detectors' unit: not a matched-burden comparison)")
L.append(f"invariants  I6 held on {len(attain)} pairs; I4 notes {sum('I4' in x for x in notes)}; "
         f"I5 notes {sum('I5' in x for x in notes)}")
_g = MEM_GUARD
L.append(f"memory  limit {str(_g['limit_gb']) + ' GB' if _g['limit_gb'] else 'unknown'} [{_g['limit_source']}] "
         f"(parsed corpus budget {str(_g['budget_gb']) + ' GB' if _g['budget_gb'] else 'none'}); university captures read "
         f"{_g['university_read']} of {_g['university_discovered']} discovered (planned {_g['university_planned']}); "
         f"guard: {_g['action']}" + (f"; one capture cost {_g['capture_cost_gb']} GB" if _g['capture_cost_gb'] else "") +
         ("; in use after each phase (GB): " + ", ".join(f"{p} {v}" for p, v in MEM_LOG) if MEM_LOG else ""))
L.append("memory limit, raw readings: " + ", ".join(f"{k}={v}" for k, v in _g["limit_raw"].items()))
if _g["action"] != "none":
    L.append(f"NOTE  the memory guard cut the university benign slice to {_g['university_read']} of "
             f"{_g['university_discovered']} captures ({_g['action']}; evenly spaced over the discovered captures, so the subset "
             "samples the whole collection rather than one stretch of it). Held-out university burdens therefore rest on "
             "fewer hours of campus traffic than a full measurement run: say so wherever they are reported.")
if IN_SAMPLE:
    L.append(f"NOTE  {sorted(IN_SAMPLE)} has a single capture here, so its burdens are in-sample (train and held-out coincide): "
             "plumbing, not evidence.")
if CELL_SECONDS:
    _top = sorted(CELL_SECONDS, key=lambda x: -x[1])[:4]
    L.append(f"time  {prov['total_seconds'] / 60:.1f} min in cells; slowest: " +
             "; ".join(f"{lab[:36].strip()} {sec / 60:.1f} min" for lab, sec in _top) +
             (f"; peak memory {prov['peak_memory_mb'] / 1024:.1f} GB" if prov.get("peak_memory_mb") else ""))
DIGEST = "\n".join(L)
open(f"{OUT_DIR}/dns_digest.txt", "w").write(DIGEST + "\n")

In [ ]:
# ================================ FIGURES ================================
_require("floors", "sweep", "attain", "ibhh")
import matplotlib as mpl, matplotlib.pyplot as plt
# Fixed categorical order, never cycled. Validated: worst adjacent CVD dE 18.7 (protan),
# normal-vision dE 26.9, all >= 3:1 on the surface. ibHH also gets a distinct marker.
PAL = {"per-query": "#1E7FA8", "flow": "#D2691E", "session": "#6A4FA3", "ibHH-style": "#3B8B3F"}
INK, MUTED, GRID = "#1A1D1F", "#5C6467", "#DDE1DE"
mpl.rcParams.update({"figure.dpi": 140, "savefig.dpi": 300, "font.size": 9,
    "axes.edgecolor": GRID, "axes.labelcolor": INK, "text.color": INK, "xtick.color": MUTED,
    "ytick.color": MUTED, "axes.spines.top": False, "axes.spines.right": False,
    "grid.color": GRID, "grid.linewidth": .6, "legend.frameon": False,
    "figure.facecolor": "white", "axes.facecolor": "white"})
def _title(ax, t): ax.set_title(t, loc="left", fontsize=9.5, fontweight="bold")
FIG_ERRORS = []

try:
    # 1 — the whole distribution, not a mean
    fig, ax = plt.subplots(figsize=(6.4, 3.9))
    for c in ORDER:
        v = np.sort(floors[floors.consumer == c].floor_measured.values * 100)
        if not len(v): continue
        y = np.arange(1, len(v) + 1) / len(v) * 100
        ax.step(v, y, where="post", color=PAL[c], lw=2, label=c)
        j = int(len(v) * .28)
        ax.annotate(c, xy=(v[j], y[j]), xytext=(7, -11), textcoords="offset points", color=PAL[c],
                    fontsize=8.5, fontweight="bold", ha="left", va="top", clip_on=False)
    ax.set_xscale("symlog", linthresh=0.01); ax.set_xlim(right=170)
    ax.set_xlabel("architectural floor: % of the asset already gone (log)")
    ax.set_ylabel("episodes at or below (%)"); ax.set_ylim(0, 101); ax.grid(alpha=.7)
    ax.set_title(f"The floor at three pipeline positions, identical traffic  (n = {len(EPISODES)})",
                 loc="left", fontsize=9.5, fontweight="bold", pad=24)
    ax.legend(loc="lower left", bbox_to_anchor=(0, 1.0), ncol=3, fontsize=8.5,
              handlelength=1.6, columnspacing=1.2, borderaxespad=.3)
    fig.tight_layout(); fig.savefig(f"{OUT_DIR}/fig_floor_ecdf.png", bbox_inches="tight"); plt.show()
except Exception as e:
    FIG_ERRORS.append('1 — the whole distribution, not a mean'); print(f'figure failed ({type(e).__name__}: {e}) — results are unaffected')

try:
    # 2 — cadence sweep with bootstrap band
    fig, axes = plt.subplots(1, 3, figsize=(12.6, 3.5), sharey=True)
    for ax, (axis, c) in zip(axes, [("flow idle timeout", "flow"), ("flow active timeout", "flow"),
                                    ("session window", "session")]):
        a = sweep[sweep.axis == axis]
        ps = sorted(a.param.unique()); mu, lo, hi = [], [], []
        for p in ps:
            v = a[a.param == p].floor_measured.values * 100
            l, h = bootstrap_ci(list(v), n_boot=2000, seed=SEED); mu.append(v.mean()); lo.append(l); hi.append(h)
        ax.fill_between(ps, lo, hi, color=PAL[c], alpha=.16, lw=0)
        ax.plot(ps, mu, color=PAL[c], lw=2, marker="o", ms=5)
        ax.set_xscale("log"); ax.set_xlabel(f"{axis} (s)"); ax.grid(alpha=.7); _title(ax, axis)
    axes[0].set_ylabel("mean floor (% of asset)")
    fig.tight_layout(); fig.savefig(f"{OUT_DIR}/fig_cadence.png", bbox_inches="tight"); plt.show()
except Exception as e:
    FIG_ERRORS.append('2 — cadence sweep with bootstrap band'); print(f'figure failed ({type(e).__name__}: {e}) — results are unaffected')

try:
    # 3 — attainment on log axes: values span 0.01% to 100%. On the diagonal = at the floor;
    #     above it = loss the detector added on top of what its pipeline position forced.
    if len(attain):
        fig, ax = plt.subplots(figsize=(5.4, 5.0))
        ax.plot([0.001, 100], [0.001, 100], color=MUTED, lw=1, ls=(0, (4, 3)), zorder=1)
        for c in ORDER:
            s_ = attain[attain.consumer == c]
            ax.scatter(s_.floor_measured * 100, s_.ebd * 100, s=30, color=PAL[c], alpha=.8,
                       edgecolor="white", linewidth=.7, label=c, zorder=3)
        if len(ibhh):
            ax.scatter(ibhh.floor_per_query * 100, ibhh.ebd * 100, s=36, marker="D",
                       color=PAL["ibHH-style"], edgecolor="white", linewidth=.7,
                       label="ibHH-style (x = per-query floor)", zorder=4)
        ax.set_xscale("symlog", linthresh=0.01); ax.set_yscale("symlog", linthresh=0.01)
        ax.set_xlim(-0.002, 150); ax.set_ylim(-0.002, 150)
        ax.annotate("EBD = floor", xy=(20, 20), xytext=(-62, 4), textcoords="offset points",
                    color=MUTED, fontsize=8)
        ax.set_xlabel("architectural floor (% of asset, log)")
        ax.set_ylabel("measured EBD (% of asset, log)"); ax.grid(alpha=.7)
        _title(ax, "Attainment: what a detector loses vs what it could not avoid")
        ax.legend(loc="lower right", fontsize=8)
        fig.tight_layout(); fig.savefig(f"{OUT_DIR}/fig_attainment.png", bbox_inches="tight"); plt.show()
except Exception as e:
    FIG_ERRORS.append('3 — attainment on log axes: values span 0.01% to 100%. On the diagonal = at the floor;'); print(f'figure failed ({type(e).__name__}: {e}) — results are unaffected')

try:
    # 4 — per family as a dot plot on a log axis: floors span three orders of magnitude,
    #     so bars on a linear axis would hide the per-query values entirely.
    fams = sorted(floors.family.unique())
    if len(fams) > 1:
        fig, ax = plt.subplots(figsize=(6.8, max(2.6, .42 * len(fams) + 1.6)))
        for i, f in enumerate(fams):
            vals = [floors[(floors.family == f) & (floors.consumer == c)].floor_measured.mean() * 100
                    for c in ORDER]
            ax.plot([min(vals), max(vals)], [i, i], color=GRID, lw=3, solid_capstyle="round", zorder=1)
            # session drawn first and larger, so a flow floor that coincides with it
            # (single-port tunnels: both 100%) still shows as a dot inside a ring
            for c, v in sorted(zip(ORDER, vals), key=lambda cv: ORDER.index(cv[0]), reverse=True):
                big = c == "session"
                ax.scatter(v, i, s=120 if big else 46, color="white" if big else PAL[c],
                           edgecolor=PAL[c] if big else "white", linewidth=2 if big else .8,
                           zorder=2 if big else 3, label=c if i == 0 else None)
        ax.set_xscale("symlog", linthresh=0.01); ax.set_xlim(-0.002, 160)
        ax.set_yticks(range(len(fams))); ax.set_yticklabels(fams, fontsize=8.5); ax.invert_yaxis()
        ax.set_xlabel("mean floor (% of asset, log)"); ax.grid(axis="x", alpha=.7)
        ax.set_title("Floor by exfiltration tool family", loc="left", fontsize=9.5,
                     fontweight="bold", pad=24)
        h_, l_ = ax.get_legend_handles_labels()
        lut_ = dict(zip(l_, h_))
        ax.legend([lut_[c] for c in ORDER if c in lut_], [c for c in ORDER if c in lut_],
                  loc="lower left", bbox_to_anchor=(0, 1.0), ncol=3, fontsize=8.5,
                  handletextpad=.3, columnspacing=1.2, borderaxespad=.3)
        fig.tight_layout(); fig.savefig(f"{OUT_DIR}/fig_by_family.png", bbox_inches="tight"); plt.show()
except Exception as e:
    FIG_ERRORS.append('4 — per family as a dot plot on a log axis: floors span three orders of magnitude,'); print(f'figure failed ({type(e).__name__}: {e}) — results are unaffected')

In [ ]:
# ======================= BUNDLE + DIGEST (send this back) =======================
_require("DIGEST")
with zipfile.ZipFile(f"{OUT_DIR}/ebd_dns_results.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for f in sorted(os.listdir(OUT_DIR)):
        if f.endswith((".csv", ".json", ".png", ".txt")):
            z.write(os.path.join(OUT_DIR, f), f)
print("=" * 100); print(DIGEST); print("=" * 100)
print(f"\nSend back: {OUT_DIR}/ebd_dns_results.zip  "
      f"({os.path.getsize(f'{OUT_DIR}/ebd_dns_results.zip')/1e6:.1f} MB) -- or paste the digest above.")

## Reading the output

**The headline this run is built to produce:** three floors on identical packets that differ
only by unit of inference. A near-zero per-query floor beside a biting flow and session floor
is Proposition 1's central claim measured rather than argued — on the one channel the
manuscript's own survey singles out as the exception, which turns the §5 scope claim into a
result.

**Check before quoting any number**

1. **SMOKE must be False** in the digest's first line. A smoke run is plumbing, not evidence.
2. **Per-capture decisions** — every included capture should show a plausible tunnel domain
   and keep most of its payload. A low "kept" share means heavy background DNS — except on a
   row marked **2-leg**, where ~50% is expected: the duplicate leg is dropped so the transfer
   is counted once.
3. **Parser line** — "captures ended early" must be 0 (otherwise the digest names the file and the
   packet it reached), and "packets skipped after an error" should be 0 or tiny. **Malformed share** —
   per tool and, separately, per benign source, with the share of the payloads sent *to* port 53 (where
   queries are) beside the share of all port-53 messages. Above 5% on a tool means non-DNS bytes on port
   53 that the asset does not count. Above 5% on a benign source means messages missing from the
   false-alarm denominator; the reasons and samples are in `dns_malformed_samples.csv`.
4. **A5 unit sensitivity** — if |delta| exceeds ~1 pp for many episodes, report floors under
   both units.
5. **AUC near 1** means the attainment cells measure the floor itself, as in the manuscript's
   §9; the realistic attainment question then rests on the university benign burden.
6. **Operating points.** Since v4.2 the threshold comes from the training-benign scores themselves, and
   the digest says what set it in each fold: *budget* (the budget bound exactly: the case the rule is meant
   to produce), *tie plateau* (equal scores at the budget's edge, so the budget is not spent in full: read
   the "budget used" share beside it), *model's zero scores* (the model, not the budget, limits
   sensitivity) or *NOT CERTIFIABLE* (fewer than 381 training units at 1%: nothing alerts, and that
   detector's numbers say nothing about the budget). Only the first is "the budget was binding". Folds
   over budget on held-out benign are reported (pooled point estimate, and per source by the Wilson
   *lower* bound, counting only folds that have held-out units from that source), never retuned on
   held-out data. The ibHH-style threshold is the largest of the per-source thresholds, so the 1% budget
   holds on each benign source; the line "pooled rule" shows what v3 would have done. The learned
   detectors' unit and ibHH's 60 s stream are different units, so the two burdens are not matched.
7. **"At floor" is strict** (|EBD − floor| < 1e-9): firing on the second query is "not at
   floor". The "within 0.01 pp" count and the max gap are the readings to quote; at family level the
   count is of families with *every* episode within 0.01 pp.
8. **Total losses depend on the exporter's timer.** A single-connection transfer is a total
   loss whenever the active timeout is at least as long as the transfer; the active-timeout
   line shows where each capture crosses over. The Chen et al. captures are 50,000-packet
   prefixes, so for them "total loss" means a transfer of that size and speed.
9. **I6 raising stops the run** — it means a bug or a false Proposition 1. Find out which.
10. **Quote the family-level lines.** Episodes of one tool differ mainly by DNS record type, so
    $n$ episodes is closer to the number of families than to $n$ independent observations. The
    digest gives the family-weighted mean with two intervals. With 8 families the **Student-t
    interval over the family means** is the one to quote (simulated coverage about 95%). The
    two-stage bootstrap (families, then episodes within a family) covers only about 90% at this
    size, so if it is quoted it is a 90% interval. Families with total loss are given as $k/m$ with a
    Wilson and an exact (Clopper-Pearson) interval. Episode-level intervals treat every episode as
    independent and are shown only for comparison.
11. **In-sample labels.** A benign source with a single capture (the university source in SMOKE) is
    both the training and the held-out set, so its burdens carry "(in-sample)" and are plumbing,
    not evidence.
12. **ibHH vs the flow floor.** An episode whose flow floor is 100% (nine of 18 on 7 Oct) is one where
    any alert before the transfer ends is "below" it. The digest gives the informative count — the
    episodes whose flow floor is under 100% — beside the raw one.

**What this licenses.** An estimate, with intervals, of how hard the floor bites across many
episodes from nine tools in eight families; a direct measurement of the pipeline-position effect; an
empirical test of the DNS carve-out; and — via the ibHH-style detector, calibrated on benign
traffic only — a measured split of loss into architectural and statistical delay. It does not
license a behavioural claim about enterprise exfiltration: the tools are lab-run and the
adversary is not adaptive. Keep DAPT2021 as the behavioural evidence and this as the
architectural evidence, the split the manuscript already draws in Limitation 1.